# DILU-IA | Análise de dados: Importar
Este caderno serve para importar os exceis resultantes dos 3 formulários Pré Sessão, Pós Sessão e Follow-Up e perpara-los para análise:

### Tabelas de output:
Serão criadas 3 tabelas para análise:

- Respostas: id x momento x exercicio, contendo respostas aos ex de conhecimento
- Escalas: id x momento, com PANAS e SIMS
- Participante: com turno, SUS score e covariáveis

Daqui resultará uma Tabela mestra para testes estatíticos: id + colunas_pré e pós.

Pasta de output: "C:\Users\Beatriz Costa\Desktop\DILU-IA"

In [1]:
#bibliotecas:
import pandas as pd
import numpy as np
import re
from itertools import combinations

In [2]:
from pathlib import Path
PASTA = Path.cwd()

## 1. Dados pre sessao

### 1.1. Importar dados pré sessão

In [3]:
df_pre = pd.read_excel(PASTA / "pre_sessao.xlsx")
df_pre

,ID,Hora de início,Hora de conclusão,E-mail,Nome,Hora da última modificação,Código de participante,Género:,Grau de escolaridade mais alto:,Qual o teu nível de conforto com a utilização de videojogos e plataformas digitais?,...,Porque é algo que eu tenho de fazer.,"Realizo esta atividade, mas não estou seguro se vale a pena.",Porque esta atividade é divertida.,Por decisão pessoal.,Porque não tenho alternativa.,"Não sei, não vejo o que é que esta atividade me pode oferecer.",Porque me sinto bem quando realizo esta atividade.,Porque acredito que esta atividade é importante para mim.,Porque sinto que tenho de fazê-lo.,"Realizo esta atividade, mas não estou seguro que seja conveniente continuar com a mesma."
0,1,2026-09-21 14:48:40,2026-09-21 14:55:38,anonymous,NaN,NaN,P043,Feminino,Ensino Secundário,3,...,3,2,5,5,1 - Não corresponde nada,3,4,5,3,4
1,2,2026-09-21 14:49:14,2026-09-21 14:55:47,anonymous,NaN,NaN,P070,Masculino,Ensino Secundário,4,...,1 - Não corresponde nada,3,5,7 - Corresponde exatamente,1 - Não corresponde nada,1 - Não corresponde nada,1 - Não corresponde nada,3,NaN,2
2,3,2026-09-21 14:49:22,2026-09-21 14:56:24,anonymous,NaN,NaN,P059,Feminino,Ensino Secundário,4,...,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente,7 - Corresponde exatamente
3,4,2026-09-21 14:49:20,2026-09-21 14:57:41,anonymous,NaN,NaN,P073,Feminino,Ensino Secundário,4,...,5,2,7 - Corresponde exatamente,4,1 - Não corresponde nada,1 - Não corresponde nada,4,7 - Corresponde exatamente,4,1 - Não corresponde nada
4,5,2026-09-21 14:48:42,2026-09-21 14:57:45,anonymous,NaN,NaN,P059,Feminino,Ensino Secundário,2,...,NaN,3,5,5,4,1 - Não corresponde nada,5,5,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,115,2026-09-21 17:47:29,2026-09-21 18:06:41,anonymous,NaN,NaN,P116,Feminino,Ensino Secundário,5,...,6,3,2,3,5,4,2,5,6,4
115,116,2026-09-21 17:42:19,2026-09-21 18:12:41,anonymous,NaN,NaN,P124,Masculino,Ensino Secundário,5,...,4,2,2,4,5,2,4,5,5,2
116,117,2026-09-21 17:42:12,2026-09-21 18:13:53,anonymous,NaN,NaN,P126,Masculino,Ensino Secundário,5,...,5,3,2,2,3,1 - Não corresponde nada,2,5,6,2
117,118,2026-09-21 17:42:07,2026-09-21 18:13:58,anonymous,NaN,NaN,P123,Masculino,Ensino Secundário,5,...,7 - Corresponde exatamente,2,2,7 - Corresponde exatamente,2,2,2,2,7 - Corresponde exatamente,1 - Não corresponde nada


In [4]:
df_pre.dtypes

ID                                                                                                                                                                                                                  int64
Hora de início                                                                                                                                                                                             datetime64[us]
Hora de conclusão                                                                                                                                                                                          datetime64[us]
E-mail                                                                                                                                                                                                                str
Nome                                                                                                                            

In [5]:
df_pre.columns.tolist()

['ID',
 'Hora de início',
 'Hora de conclusão',
 'E-mail',
 'Nome',
 'Hora da última modificação',
 'Código de participante',
 'Género:',
 'Grau de escolaridade mais alto:',
 'Qual o teu nível de conforto com a utilização de videojogos e plataformas digitais?',
 'Qual o teu nível de conforto com a utilização de ferramentas de Inteligência Artificial?',
 'Precisas de preparar 250 mL de uma solução de sulfato de magnésio (MgSO₄) com concentração 15 mM. Sabendo que a massa molar do MgSO₄ é 120,37 g/mol, que massa (em mg) deves pesar?',
 'Qual a estratégia de resolução que utilizaste?',
 'Uma solução concentrada de KCl a 2,5 M precisa de ser diluída até 0,6 M antes da perfusão. Partindo de 15 mL da solução concentrada, que volume de soro deves adicionar?',
 'Qual a estratégia de resolução que utilizaste?2',
 'Para determinar a concentração mínima inibitória (CMI) de um antibiótico, preparaste 3 diluições seriadas sucessivas, cada uma com fator 1:5. A concentração testada na última diluição

### 1.2. Limpeza inicial

1. Removidas colunas administrativas
2. Colunas renomeadas
3. Conversão PANAS e SIMS para valores numéricos
4. Conversão variáveis demográficas
5. Uniformização codigos de participante
6. Cálculo do tempo de preenchimento do questionário
7. Identificação do numero de submissões para o fluxograma

In [6]:
# Remover colunas
df_pre.drop(columns=['ID', 'E-mail', 'Nome', 'Hora da última modificação'],
            inplace=True, errors='ignore')

In [7]:
#Renomear colunas

new_names_pre = {
    'Hora de início' : 'start_time',
    'Hora de conclusão' : 'end_time',
    'Código de participante' : 'participant_id',
    'Género:' : 'gender' ,
    'Grau de escolaridade mais alto:' : 'education',
    'Qual o teu nível de conforto com a utilização de videojogos e plataformas digitais?' : 'digital_confort',
    'Qual o teu nível de conforto com a utilização de ferramentas de Inteligência Artificial?' : 'ia_confort',
    # Conhecimento
    'Precisas de preparar 250 mL de uma solução de sulfato de magnésio (MgSO₄) com concentração 15 mM. Sabendo que a massa molar do MgSO₄ é 120,37 g/mol, que massa (em mg) deves pesar?' : 'ex1_answer',
    'Qual a estratégia de resolução que utilizaste?' : 'ex1_strategy',
    'Uma solução concentrada de KCl a 2,5 M precisa de ser diluída até 0,6 M antes da perfusão. Partindo de 15 mL da solução concentrada, que volume de soro deves adicionar?' : 'ex2_answer',
    'Qual a estratégia de resolução que utilizaste?2' : 'ex2_strategy',
    'Para determinar a concentração mínima inibitória (CMI) de um antibiótico, preparaste 3 diluições seriadas sucessivas, cada uma com fator 1:5. A concentração testada na última diluição foi de 8 µg/...' : 'ex3_answer',
    'Qual a estratégia de resolução que utilizaste?3' : 'ex3_strategy',
    # PANAS-VPR pre 
    'Interessado/a' : 'affect_1',
    'Nervoso/a' : 'affect_2',
    'Entusiasmado/a' : 'affect_3',
    'Amedrontado/a' : 'affect_4',
    'Inspirado/a' : 'affect_5',
    'Ativo/a' : 'affect_6',
    'Assustado/a' : 'affect_7',
    'Culpado/a' : 'affect_8',
    'Determinado/a' : 'affect_9',
    'Atormentado/a' : 'affect_10',
    #SIMS pre (Items 1-6)
    'Porque penso que esta atividade é interessante.' : 'motiv_1',
    'Porque é para o meu próprio bem.' : 'motiv_2',
    'Porque é suposto em fazê-lo.' : 'motiv_3',
    'Pode ser que existam boas razões para realizar esta atividade, mas pessoalmente não vejo nenhuma.' : 'motiv_4',
    'Porque considero que esta atividade é agradável.' : 'motiv_5',
    'Porque penso que esta atividade é boa para mim.' : 'motiv_6',
    'Porque é algo que eu tenho de fazer.' : 'motiv_7',
    'Realizo esta atividade, mas não estou seguro se vale a pena.' : 'motiv_8',
    'Porque esta atividade é divertida.' : 'motiv_9',
    'Por decisão pessoal.' : 'motiv_10',
    'Porque não tenho alternativa.' : 'motiv_11',
    'Não sei, não vejo o que é que esta atividade me pode oferecer.' : 'motiv_12',
    'Porque me sinto bem quando realizo esta atividade.' : 'motiv_13',
    'Porque acredito que esta atividade é importante para mim.' : 'motiv_14',
    'Porque sinto que tenho de fazê-lo.' : 'motiv_15',
    'Realizo esta atividade, mas não estou seguro que seja conveniente continuar com a mesma.' : 'motiv_16'
}

df_pre = df_pre.rename(columns=new_names_pre, errors='ignore')
print("Colunas renomeadas.")

Colunas renomeadas.


In [8]:
# Converter itens PANAS e SIMS para numérico
# Alguns itens chegam como texto com o número no início (ex: "4 - Concordo")
cols_numericas = [c for c in df_pre.columns if c.startswith(('affect_', 'motiv_'))]
for col in cols_numericas:
     if not pd.api.types.is_numeric_dtype(df_pre[col]):
        df_pre[col] = pd.to_numeric(
            df_pre[col].astype(str).str.extract(r'^(\d+)', expand=False),
            errors='coerce'
        )

print(f"Colunas convertidas para numérico: {len(cols_numericas)}")

Colunas convertidas para numérico: 26


In [9]:
df_pre['education'].value_counts()

education
Ensino Secundário    112
Licenciatura           7
Name: count, dtype: int64

In [10]:
# Variáveis demográficas
df_pre['prior_education'] = df_pre['education'].astype(str).map({'Ensino Secundário' : 0, 'Licenciatura' : 1})

GENERO = {"Feminino": 0, "Masculino": 1}
assert df_pre["gender"].isin(GENERO.keys()).all(), \
    f"Há respostas de género fora de Feminino/Masculino: {df_pre['gender'].unique()}"
df_pre["gender"] = df_pre["gender"].map(GENERO)

# Normalizar participant_id (uppercase, remover espaços)
df_pre['participant_id'] = (df_pre['participant_id'].astype(str).str.extract(r'(\d+)', expand = False).astype(int))
df_pre.sort_values('participant_id', inplace=True)

# Calcular tempo de preenchimento
df_pre['fill_in_time'] = ((df_pre['end_time'] - df_pre['start_time'])
                                   .dt.total_seconds() / 60).round(2)

print(f"Dataset pré processado: {df_pre.shape}")
df_pre.head()

Dataset pré processado: (119, 41)


,start_time,end_time,participant_id,gender,education,digital_confort,ia_confort,ex1_answer,ex1_strategy,ex2_answer,...,motiv_9,motiv_10,motiv_11,motiv_12,motiv_13,motiv_14,motiv_15,motiv_16,prior_education,fill_in_time
21,2026-09-21 14:48:42,2026-09-21 15:06:02,1,0,Ensino Secundário,2,2,451388.00,Fórmula matemática,45.00,...,4.0,3.0,3.0,4.0,3.0,4.0,3.0,3.0,0,17.33
8,2026-09-21 14:49:21,2026-09-21 15:02:29,2,0,Ensino Secundário,5,5,451.00,Fórmula matemática,48.00,...,7.0,1.0,6.0,1.0,7.0,7.0,6.0,1.0,0,13.13
31,2026-09-21 14:49:02,2026-09-21 15:07:34,4,0,Ensino Secundário,4,3,3.75,Fórmula matemática,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,18.53
29,2026-09-21 14:49:21,2026-09-21 15:07:29,5,1,Ensino Secundário,3,4,451.00,Fórmula matemática,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,18.13
59,2026-09-21 16:15:12,2026-09-21 16:34:11,7,0,Ensino Secundário,4,3,451388.00,Desenhei um esquema,6.25,...,6.0,4.0,4.0,2.0,4.0,6.0,6.0,3.0,0,18.98


In [11]:
fluxogram = {}
fluxogram["pre_submissions"] = len(df_pre)
print(fluxogram)

{'pre_submissions': 119}


### 1.3. Identificação de duplicados para decisão posterior

In [12]:
#Sinalizar duplicados para decisão caso a caso antes do merge com dados pos
pre_duplicados = df_pre[df_pre["participant_id"].duplicated(keep=False)]
pre_duplicados = pre_duplicados.sort_values(["participant_id", "start_time"])
pre_duplicados[["participant_id", "start_time", "end_time", "gender", "education", 'digital_confort', 'ia_confort', 'fill_in_time']]

,participant_id,start_time,end_time,gender,education,digital_confort,ia_confort,fill_in_time
73,10,2026-09-21 16:17:45,2026-09-21 16:42:27,0,Ensino Secundário,4,5,24.70
78,10,2026-09-21 16:45:47,2026-09-21 16:53:59,0,Ensino Secundário,5,5,8.20
4,59,2026-09-21 14:48:42,2026-09-21 14:57:45,0,Ensino Secundário,2,3,9.05
2,59,2026-09-21 14:49:22,2026-09-21 14:56:24,0,Ensino Secundário,4,3,7.03
51,113,2026-09-21 16:15:37,2026-09-21 16:29:18,0,Ensino Secundário,4,3,13.68
83,113,2026-09-21 17:41:23,2026-09-21 17:50:52,1,Ensino Secundário,4,4,9.48


## 2. Dados pós sessão


### 2.1. Importar dados pós sessão

In [13]:
df_pos = pd.read_excel(PASTA / "pos_sessao.xlsx")
df_pos

,ID,Hora de início,Hora de conclusão,E-mail,Nome,Hora da última modificação,Código de participante,Cola aqui as teus resultados do jogo:,"Durante o jogo, o LabPartner:","Precisas de preparar 300 mL de uma solução de bicarbonato de sódio (NaHCO₃), com concentração 20 mM. Sabendo que a massa molar do NaHCO₃ é 84,01 g/mol, que massa (em mg) deves pesar?",...,Acho que necessitaria da ajuda de um técnico para conseguir utilizar este jogo.,Considerei que as várias funcionalidades deste jogo estavam bem integradas.,Achei que este jogo tinha demasiadas inconsistências.,Suponho que a maioria das pessoas aprenderia a utilizar rapidamente este jogo.,Considerei o jogo muito complicado de utilizar.,Senti-me muito confiante a utilizar este jogo.,Tive que aprender muito antes de conseguir lidar com o jogo.,Obrigada pela participação! Deixa aqui comentários gerais sobre a experiência:,Quais foram os principais aspetos positivos do jogo?,Quais foram os principais aspetos negativos ou limitações do jogo?
0,1,2026-09-21 15:42:58,2026-09-21 15:46:00,anonymous,NaN,NaN,P059,NaN,Funcionou bem (nada de estranho a reportar);,504.0,...,3,3,3,3,3,3,3,NaN,Iterativo,Não teve
1,2,2026-09-21 15:42:51,2026-09-21 15:47:58,anonymous,NaN,NaN,P032,NaN,Nao consegui usar;,504.0,...,1,4,3,4,3,3,1,NaN,Ajuda,Nao consegui entrar
2,3,2026-09-21 15:44:02,2026-09-21 15:48:06,anonymous,NaN,NaN,P030,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,...,1,5,1,5,1,5,1,NaN,Bem estruturado,Nao vi nenhumas
3,4,2026-09-21 15:46:36,2026-09-21 15:48:20,anonymous,NaN,NaN,P058,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,6.0,...,5,5,5,5,5,5,5,NaN,Fácil de usar,Nenhum
4,5,2026-09-21 15:42:48,2026-09-21 15:48:49,anonymous,NaN,NaN,P031,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Não funcionou;,504.0,...,1,4,1,3,1,2,1,NaN,Intuitividade,O labpartner nao funcionou
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,89,2026-09-21 18:42:02,2026-09-21 18:48:13,anonymous,NaN,NaN,P147,NaN,Funcionou bem (nada de estranho a reportar);,504.0,...,2,5,1,5,1,4,1,NaN,Rapidez,Não tenho
89,90,2026-09-21 18:43:00,2026-09-21 18:48:47,anonymous,NaN,NaN,P118,NaN,Deu-me o código antes de eu resolver os exercí...,540.0,...,2,5,1,3,1,3,1,NaN,A ajuda do chat,nenhuns
90,91,2026-09-21 18:32:40,2026-09-21 19:01:25,anonymous,NaN,NaN,P146,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Nunca usei;,504.0,...,2,3,3,4,2,4,2,NaN,Apresentar cálculos de maneira simples,Escrever os cálculos no ecrã
91,92,2026-09-21 15:52:33,2026-09-21 20:32:28,anonymous,NaN,NaN,P005,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,...,2,5,1,5,2,3,1,NaN,Gostei do facto de poder estudar autonomamente...,Acho que o tempo que tivemos foi muito reduzid...


In [14]:
df_pos.columns.tolist()

['ID',
 'Hora de início',
 'Hora de conclusão',
 'E-mail',
 'Nome',
 'Hora da última modificação',
 'Código de participante',
 'Cola aqui as teus resultados do jogo:',
 'Durante o jogo, o LabPartner:',
 'Precisas de preparar 300 mL de uma solução de bicarbonato de sódio (NaHCO₃), com concentração 20 mM. Sabendo que a massa molar do NaHCO₃ é 84,01 g/mol, que massa (em mg) deves pesar?',
 'Qual a estratégia de resolução que utilizaste?',
 'Uma solução hipertónica de NaCl a 3 M precisa de ser diluída até 0,75 M antes da administração. Partindo de 20 mL da solução concentrada, que volume de soro deves adicionar?',
 'Qual a estratégia de resolução que utilizaste?2',
 'Para titular um soro com anticorpos, preparaste 3 diluições seriadas sucessivas, cada uma com fator 1:4. A diluição final testada correspondia a uma concentração de 15 U/mL. Qual era a concentraçã...',
 'Qual a estratégia de resolução que utilizaste?3',
 'Interessado/a',
 'Nervoso/a',
 'Entusiasmado/a',
 'Amedrontado/a',
 'Ins

In [15]:
df_pos.dtypes

ID                                                                                                                                                                                                                  int64
Hora de início                                                                                                                                                                                             datetime64[us]
Hora de conclusão                                                                                                                                                                                          datetime64[us]
E-mail                                                                                                                                                                                                                str
Nome                                                                                                                            

### 2.2. Limpeza inicial pos sessão

1. Removidas colunas administrativas
2. Colunas renomeadas
3. Conversão PANAS e SIMS para valores numéricos
5. Uniformização codigos de participante
6. Cálculo do tempo de preenchimento do questionário
7. Criação de colunas relacionadas com game stats do jogo
8. Identificação do numero de submissões para o fluxograma

In [16]:
# Remover colunas
df_pos.drop(columns=['ID', 'E-mail', 'Nome', 'Hora da última modificação'],
            inplace=True, errors='ignore')

In [17]:
#Renomear colunas

new_names_pos = {
    'Hora de início' : 'start_time',
    'Hora de conclusão' : 'end_time',
    'Código de participante' : 'participant_id',
    'Cola aqui as teus resultados do jogo:' : 'game_results',
    'Durante o jogo, o LabPartner:' : 'labpartner_function',
    # Conhecimento
    'Precisas de preparar 300 mL de uma solução de bicarbonato de sódio (NaHCO₃), com concentração 20 mM. Sabendo que a massa molar do NaHCO₃ é 84,01 g/mol, que massa (em mg) deves pesar?' : 'ex1_answer',
    'Qual a estratégia de resolução que utilizaste?' : 'ex1_strategy',
    'Uma solução hipertónica de NaCl a 3 M precisa de ser diluída até 0,75 M antes da administração. Partindo de 20 mL da solução concentrada, que volume de soro deves adicionar?' : 'ex2_answer',
    'Qual a estratégia de resolução que utilizaste?2' : 'ex2_strategy',
    'Para titular um soro com anticorpos, preparaste 3 diluições seriadas sucessivas, cada uma com fator 1:4. A diluição final testada correspondia a uma concentração de 15 U/mL. Qual era a concentraçã...' : 'ex3_answer',
    'Qual a estratégia de resolução que utilizaste?3' : 'ex3_strategy',
    # PANAS-VPR pos 
    'Interessado/a' : 'affect_1',
    'Nervoso/a' : 'affect_2',
    'Entusiasmado/a' : 'affect_3',
    'Amedrontado/a' : 'affect_4',
    'Inspirado/a' : 'affect_5',
    'Ativo/a' : 'affect_6',
    'Assustado/a' : 'affect_7',
    'Culpado/a' : 'affect_8',
    'Determinado/a' : 'affect_9',
    'Atormentado/a' : 'affect_10',
    #SIMS pos (Items 1-6)
    'Porque penso que esta atividade é interessante.' : 'motiv_1',
    'Porque é para o meu próprio bem.' : 'motiv_2',
    'Porque é suposto em fazê-lo.' : 'motiv_3',
    'Pode ser que existam boas razões para realizar esta atividade, mas pessoalmente não vejo nenhuma.' : 'motiv_4',
    'Porque considero que esta atividade é agradável.' : 'motiv_5',
    'Porque penso que esta atividade é boa para mim.' : 'motiv_6',
    'Porque é algo que eu tenho de fazer.' : 'motiv_7',
    'Realizo esta atividade, mas não estou seguro se vale a pena.' : 'motiv_8',
    'Porque esta atividade é divertida.' : 'motiv_9',
    'Por decisão pessoal.' : 'motiv_10',
    'Porque não tenho alternativa.' : 'motiv_11',
    'Não sei, não vejo o que é que esta atividade me pode oferecer.' : 'motiv_12',
    'Porque me sinto bem quando realizo esta atividade.' : 'motiv_13',
    'Porque acredito que esta atividade é importante para mim.' : 'motiv_14',
    'Porque sinto que tenho de fazê-lo.' : 'motiv_15',
    'Realizo esta atividade, mas não estou seguro que seja conveniente continuar com a mesma.' : 'motiv_16',
    #SUS
    'Acho que gostaria de utilizar este jogo com frequência.' : 'sus1',
    'Considerei o jogo mais complexo do que necessário.' : 'sus2',
    'Achei o jogo fácil de utilizar.' : 'sus3',
    'Acho que necessitaria da ajuda de um técnico para conseguir utilizar este jogo.' : 'sus4',
    'Considerei que as várias funcionalidades deste jogo estavam bem integradas.' : 'sus5',
    'Achei que este jogo tinha demasiadas inconsistências.' : 'sus6',
    'Suponho que a maioria das pessoas aprenderia a utilizar rapidamente este jogo.' : 'sus7',
    'Considerei o jogo muito complicado de utilizar.' : 'sus8',
    'Senti-me muito confiante a utilizar este jogo.' : 'sus9',
    'Tive que aprender muito antes de conseguir lidar com o jogo.' : 'sus10',
    ##Comentários abertos
    '\xa0Obrigada pela participação! Deixa aqui comentários gerais sobre a experiência:' : 'general_comment',
    'Quais foram os principais aspetos positivos do jogo?' : 'positive_comment',
    'Quais foram os principais aspetos negativos ou limitações do jogo?' : 'negative_comment'
}

df_pos = df_pos.rename(columns=new_names_pos, errors='ignore')
print("Colunas renomeadas.")

Colunas renomeadas.


In [18]:
# Converter itens PANAS e SIMS para numérico
# Alguns itens chegam como texto com o número no início (ex: "4 - Concordo")
cols_numericas = [c for c in df_pos.columns if c.startswith(('affect_', 'motiv_'))]
for col in cols_numericas:
     if not pd.api.types.is_numeric_dtype(df_pos[col]):
        df_pos[col] = pd.to_numeric(
            df_pos[col].astype(str).str.extract(r'^(\d+)', expand=False),
            errors='coerce'
        )

print(f"Colunas convertidas para numérico: {len(cols_numericas)}")

Colunas convertidas para numérico: 26


In [19]:
# Normalizar participant_id (uppercase, remover espaços)
df_pos['participant_id'] = (df_pos['participant_id'].astype(str).str.extract(r'(\d+)', expand = False).astype(int))
df_pos.sort_values('participant_id', inplace=True)

# Calcular tempo de preenchimento
df_pos['fill_in_time'] = ((df_pos['end_time'] - df_pos['start_time'])
                                   .dt.total_seconds() / 60).round(2)

print(f"Dataset pré processado: {df_pos.shape}")
df_pos.head()

Dataset pré processado: (93, 51)


,start_time,end_time,participant_id,game_results,labpartner_function,ex1_answer,ex1_strategy,ex2_answer,ex2_strategy,ex3_answer,...,sus5,sus6,sus7,sus8,sus9,sus10,general_comment,positive_comment,negative_comment,fill_in_time
7,2026-09-21 15:47:44,2026-09-21 15:50:21,1,NaN,Deu respostas confusas ou contraditórias;,0.0,Tentativa e erro,0,Tentativa e erro,0.0,...,2,2,2,1,2,4,NaN,Ajuda aprender,É complicado,2.62
8,2026-09-21 15:43:59,2026-09-21 15:50:36,2,XP: 1550\nRank: cientista chefe \nEstabilidade...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,80,Fórmula matemática,180.0,...,5,1,5,1,5,1,NaN,Todos,Nenhuns,6.62
91,2026-09-21 15:52:33,2026-09-21 20:32:28,5,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,60,Fórmula matemática,960.0,...,5,1,5,2,3,1,NaN,Gostei do facto de poder estudar autonomamente...,Acho que o tempo que tivemos foi muito reduzid...,279.92
52,2026-09-21 17:14:33,2026-09-21 17:21:19,7,💥 DILU-IA - Explosão no Lab!\n\n📊 Resultados:\...,Funcionou bem (nada de estranho a reportar);,1.0,Fórmula matemática,80,Fórmula matemática,0.0,...,3,3,3,3,3,3,Gostei mas foi cansativo,Aprender,O tempo reduzido,6.77
24,2026-09-21 15:43:07,2026-09-21 15:53:35,8,💥 DILU-IA - Explosão no Lab!\n\n📊 Resultados:\...,Funcionou bem (nada de estranho a reportar);,100.0,Fórmula matemática,0,não sei,0.0,...,5,1,5,1,4,2,NaN,bastante intuitivo,nada a referir,10.47


In [20]:
#Separar vitorias, game overs e outros reports
def win_lose(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return np.nan
    texto = str(texto)
    if "Laboratório Salvo" in texto:
        return 1
    if "Explosão no Lab" in texto:
        return 0
    return "outro"

df_pos["game_finished"] = df_pos["game_results"].apply(win_lose)
print(df_pos["game_finished"].value_counts())

with pd.option_context('display.max_colwidth', None):
    display(
        df_pos.loc[
            df_pos["game_finished"] == "outro",
            [ "game_results"]
        ]
    )

game_finished
1        33
0        30
outro    13
Name: count, dtype: int64


,game_results
8,"XP: 1550\nRank: cientista chefe \nEstabilidade: 100% \nTempo: 32m49s\nNivel 1: 3/3,Nivel 2: 2/3, nivel 3:1/3"
70,"3,75,1000,100"
29,estabilidade: 70%
27,"Passei em ambos os níveis, completando os 3 jogos disponíveis. No entanto, quando estava a inserir o código para passar ao nível seguinte( lab2026), a página fechou."
77,"✨\nXP Total: 1550\n🏆\nRank: Cientista Chefe 🧪\n⚗️\nEstabilidade Final: 90%\n⏱️\nTempo: 30m 17s\n🤖\nUso do LabPartner: Nível 1: 2/3, Nível 2: 0/3, Nível 3: 0/3"
64,Não consegui copiar os resultados
57,Nível 1
48,800
65,1550xp\nEstabilidade final95%
66,1550 xp total\nEstabilidade final 100%\n21m 36s


In [21]:
# extraindo os valores do texto de game_results
texto = df_pos["game_results"].astype("string")

df_pos["xp"] = texto.str.extract(r"XP(?: Total)?:\s*(\d+)", flags=re.IGNORECASE)[0].astype(float)
df_pos["rank_txt"] = texto.str.extract(r"Rank:\s*([A-Za-zÀ-ÿ ]+)", flags=re.IGNORECASE)[0].str.strip().str.lower()
df_pos["estabilidade"] = texto.str.extract(r"Estabilidade(?: Final)?:?\s*(\d+)%", flags=re.IGNORECASE)[0].astype(float)

tempo = texto.str.extract(r"(\d+)m\s*(\d+)s")
df_pos["game_time_min"] = tempo[0].astype(float) + tempo[1].astype(float) / 60

for n in [1, 2, 3]:
    df_pos[f"lp_level{n}"] = texto.str.extract(rf"N[ií]vel {n}:\s*(\d)/3", flags=re.IGNORECASE)[0].astype(float)

RANK_PARA_NIVEL = {"estagiário": 1, "técnico de lab": 2, "cientista chefe": 3}
df_pos["final_level"] = df_pos["rank_txt"].map(RANK_PARA_NIVEL)

with pd.option_context('display.max_colwidth', None):
    display(
        df_pos.loc[
            df_pos["game_finished"] == "outro",
            [ "game_results", "xp", "rank_txt", "estabilidade", "game_time_min", "lp_level1", "lp_level2", "lp_level3", "final_level"]
        ])

,game_results,xp,rank_txt,estabilidade,game_time_min,lp_level1,lp_level2,lp_level3,final_level
8,"XP: 1550\nRank: cientista chefe \nEstabilidade: 100% \nTempo: 32m49s\nNivel 1: 3/3,Nivel 2: 2/3, nivel 3:1/3",1550.0,cientista chefe,100.0,32.816667,3.0,2.0,1.0,3.0
70,"3,75,1000,100",NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
29,estabilidade: 70%,NaN,<NA>,70.0,NaN,NaN,NaN,NaN,NaN
27,"Passei em ambos os níveis, completando os 3 jogos disponíveis. No entanto, quando estava a inserir o código para passar ao nível seguinte( lab2026), a página fechou.",NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
77,"✨\nXP Total: 1550\n🏆\nRank: Cientista Chefe 🧪\n⚗️\nEstabilidade Final: 90%\n⏱️\nTempo: 30m 17s\n🤖\nUso do LabPartner: Nível 1: 2/3, Nível 2: 0/3, Nível 3: 0/3",1550.0,cientista chefe,90.0,30.283333,2.0,0.0,0.0,3.0
64,Não consegui copiar os resultados,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
57,Nível 1,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
48,800,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
65,1550xp\nEstabilidade final95%,NaN,<NA>,95.0,NaN,NaN,NaN,NaN,NaN
66,1550 xp total\nEstabilidade final 100%\n21m 36s,NaN,<NA>,100.0,21.600000,NaN,NaN,NaN,NaN


In [22]:
df_pos["nota_jogo"] = ""

CLASSIFICACAO_MANUAL = {
    #indice : {coluna: valor, "nota_jogo": "motivo"},
    8 : {"game_finished" : 1, "nota_jogo" : "sem cabeçalho; XP 1550"},
    70 : {"game_finished" : np.nan, "nota_jogo" : "sem dados"},
    29 : {"game_finished" : np.nan, "nota_jogo" : "sem dados"},
    27 : {"game_finished" : 0, "final_level" : 2, "nota_jogo" : "completou níveis 1 e 2; usou o código do nível 1 no fim do nível 2 e a página fechou"},
    77 : {"game_finished" : 1, "nota_jogo" : "sem cabeçalho; XP 1550"},
    64 : {"game_finished" : np.nan, "nota_jogo" : "nao conseguiu copiar"},
    57 : {"game_finished" : 0 , "final_level" : 1, "nota_jogo" : "não terminou nem conseguiu copiar; a pedido da investigadora indicou o nível em que estava (1)"},
    48 : {"game_finished" : np.nan, "nota_jogo" : "sem dados"},
    65 : {"game_finished" : 1, "xp" : 1550, "nota_jogo" : "texto mal copiado"},
    66 : {"game_finished" : 1, "xp" : 1550, "rank_txt" : "cientista chefe", "estabilidade" : 100, "final_level" : 3, "nota_jogo" : "texto mal copiado"},
    71 : {"game_finished" : np.nan, "nota_jogo" : "sem dados"},
    67 : {"game_finished" : np.nan, "nota_jogo" : "sem dados"},
    87 : {"game_finished" : np.nan, "nota_jogo" : "nao conseguiu copiar"}
}


   
for indice, valores in CLASSIFICACAO_MANUAL.items():
    for coluna, valor in valores.items():
        df_pos.loc[indice, coluna] = valor

so_numeros = df_pos["game_finished"].isin([0, 1]) | df_pos["game_finished"].isna()
assert so_numeros.all(), "Ainda há texto em game_finished (casos 'outro' por classificar?)"

df_pos["game_finished"] = df_pos["game_finished"].astype(float)
df_pos["game_finished"].value_counts(dropna=False)

game_finished
1.0    37
0.0    32
NaN    24
Name: count, dtype: int64

In [23]:
df_pos.loc[df_pos["game_finished"] == 1, "final_level"] = 3
pd.crosstab(df_pos["final_level"], df_pos["game_finished"], dropna=False)

game_finished,0.0,1.0,NaN
final_level,,,
1.0,5,0,0
2.0,16,0,0
3.0,11,37,0
NaN,0,0,24


A proporção que chegou ao nível 3 está entre dois limites:

no mínimo 48 em 93 (52%), se nenhum dos 24 tiver chegado;
no máximo 72 em 93 (77%), se todos tiverem chegado.

In [24]:
df_pos["arrived_level3"] = (df_pos["final_level"] == 3).astype(float)
df_pos.loc[df_pos["final_level"].isna(), "arrived_level3"] = np.nan

In [25]:
df_pos

,start_time,end_time,participant_id,game_results,labpartner_function,ex1_answer,ex1_strategy,ex2_answer,ex2_strategy,ex3_answer,...,xp,rank_txt,estabilidade,game_time_min,lp_level1,lp_level2,lp_level3,final_level,nota_jogo,arrived_level3
7,2026-09-21 15:47:44,2026-09-21 15:50:21,1,NaN,Deu respostas confusas ou contraditórias;,0.0,Tentativa e erro,0,Tentativa e erro,0.00,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
8,2026-09-21 15:43:59,2026-09-21 15:50:36,2,XP: 1550\nRank: cientista chefe \nEstabilidade...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,80,Fórmula matemática,180.00,...,1550.0,cientista chefe,100.0,32.816667,3.0,2.0,1.0,3.0,sem cabeçalho; XP 1550,1.0
91,2026-09-21 15:52:33,2026-09-21 20:32:28,5,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,60,Fórmula matemática,960.00,...,1550.0,cientista chefe,100.0,41.850000,3.0,1.0,1.0,3.0,,1.0
52,2026-09-21 17:14:33,2026-09-21 17:21:19,7,💥 DILU-IA - Explosão no Lab!\n\n📊 Resultados:\...,Funcionou bem (nada de estranho a reportar);,1.0,Fórmula matemática,80,Fórmula matemática,0.00,...,600.0,técnico de lab,0.0,16.600000,3.0,1.0,0.0,2.0,,0.0
24,2026-09-21 15:43:07,2026-09-21 15:53:35,8,💥 DILU-IA - Explosão no Lab!\n\n📊 Resultados:\...,Funcionou bem (nada de estranho a reportar);,100.0,Fórmula matemática,0,não sei,0.00,...,300.0,técnico de lab,0.0,31.833333,3.0,0.0,0.0,2.0,,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,2026-09-21 18:41:08,2026-09-21 18:46:13,145,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,60,Fórmula matemática,960.00,...,1050.0,cientista chefe,100.0,14.333333,1.0,0.0,2.0,3.0,,1.0
90,2026-09-21 18:32:40,2026-09-21 19:01:25,146,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Nunca usei;,504.0,Regra de três simples,0,Regra de três simples,960.00,...,1050.0,cientista chefe,100.0,35.083333,0.0,0.0,0.0,3.0,,1.0
88,2026-09-21 18:42:02,2026-09-21 18:48:13,147,NaN,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,6,Fórmula matemática,3.75,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
86,2026-09-21 18:41:27,2026-09-21 18:46:58,148,🎉 DILU-IA - Laboratório Salvo!\n\n📊 Resultados...,Funcionou bem (nada de estranho a reportar);,504.0,Fórmula matemática,60,Fórmula matemática,960.00,...,1050.0,cientista chefe,85.0,33.016667,0.0,0.0,1.0,3.0,,1.0


In [26]:
fluxogram["pos_submissions"] = len(df_pos)

In [27]:
print(fluxogram)

{'pre_submissions': 119, 'pos_submissions': 93}


In [28]:
OPCOES_LP = {
    "Deu-me o código antes de eu resolver os exercícios": "lp_deu_codigo",
    "Fez cálculos sem eu explicar o raciocínio": "lp_fez_calculos",
    "Deu respostas confusas ou contraditórias": "lp_confuso",
    "Funcionou bem (nada de estranho a reportar)": "lp_funcionou_bem",
}

texto_lp = df_pos["labpartner_function"].fillna("")
for opcao, coluna in OPCOES_LP.items():
    df_pos[coluna] = texto_lp.str.contains(opcao, regex=False).astype(float)

In [29]:
def texto_outro_lp(texto):
    if pd.isna(texto):
        return np.nan
    partes = [p.strip() for p in str(texto).split(";") if p.strip()]
    outras = [p for p in partes if p not in OPCOES_LP]
    return "; ".join(outras) if outras else np.nan

df_pos["lp_outro_texto"] = df_pos["labpartner_function"].apply(texto_outro_lp)
df_pos["lp_outro"] = df_pos["lp_outro_texto"].notna().astype(float)

In [30]:
colunas_lp = list(OPCOES_LP.values()) + ["lp_outro"]
df_pos.loc[df_pos["labpartner_function"].isna(), colunas_lp] = np.nan

df_pos["lp_problema"] = df_pos[["lp_deu_codigo", "lp_fez_calculos", "lp_confuso"]].max(axis=1)

In [31]:
sem_opcao = df_pos["labpartner_function"].notna() & (df_pos[colunas_lp].sum(axis=1) == 0)
assert not sem_opcao.any(), "Há respostas sem nenhuma opção reconhecida"

incoerente = (df_pos["lp_funcionou_bem"] == 1) & (df_pos["lp_problema"] == 1)
print("Funcionou bem + problema:", incoerente.sum())

print(df_pos[colunas_lp + ["lp_problema"]].sum())
df_pos.loc[df_pos["lp_outro"] == 1, ["participant_id", "lp_outro_texto"]]

Funcionou bem + problema: 2
lp_deu_codigo        1.0
lp_fez_calculos      1.0
lp_confuso           8.0
lp_funcionou_bem    68.0
lp_outro            17.0
lp_problema         10.0
dtype: float64


,participant_id,lp_outro_texto
4,31,Não funcionou
1,32,Nao consegui usar
51,35,Não funcionou
26,57,Não usei
34,63,nada
60,64,Não foi utilizado
54,66,Não funcionou
77,75,Não consegui usar- aparece que sim mas não
39,92,Nao funcionou bem
55,97,Nao usei


In [32]:
def codificar_outro_lp(texto):
    if pd.isna(texto):
        return np.nan
    t = str(texto).lower()
    if "funcion" in t or "consegui" in t or "lento" in t:
        return "problema_tecnico"
    if "us" in t or "utiliz" in t:
        return "nao_usou"
    return "outro"

df_pos["lp_outro_cat"] = df_pos["lp_outro_texto"].apply(codificar_outro_lp)

In [33]:
pd.crosstab(df_pos["lp_outro_texto"], df_pos["lp_outro_cat"])

lp_outro_cat,nao_usou,outro,problema_tecnico
lp_outro_texto,,,
Foi muito lento,0,0,1
Nao consegui usar,0,0,1
Nao funcionou bem,0,0,1
Nao usei,1,0,0
Nunca usei,1,0,0
Não consegui usar- aparece que sim mas não,0,0,1
Não foi utilizado,1,0,0
Não funcionou,0,0,4
Não usei,4,0,0


In [34]:
df_pos["lp_problema_tecnico"] = (df_pos["lp_outro_cat"] == "problema_tecnico").astype(float)
df_pos["lp_nao_usou"] = (df_pos["lp_outro_cat"] == "nao_usou").astype(float)
df_pos["lp_outro"] = (df_pos["lp_outro_cat"] == "outro").astype(float)

colunas_lp_novas = ["lp_problema_tecnico", "lp_nao_usou", "lp_outro"]
df_pos.loc[df_pos["labpartner_function"].isna(), colunas_lp_novas] = np.nan

In [35]:
df_pos.loc[df_pos["lp_nao_usou"] == 1, ["participant_id", "lp_level1", "lp_level2", "lp_level3"]]

,participant_id,lp_level1,lp_level2,lp_level3
26,57,2.0,0.0,0.0
60,64,2.0,0.0,0.0
55,97,1.0,0.0,0.0
58,99,NaN,NaN,NaN
78,116,1.0,0.0,0.0
79,131,1.0,0.0,0.0
66,134,NaN,NaN,NaN
90,146,0.0,0.0,0.0


In [36]:
print(df_pos[["lp_deu_codigo", "lp_fez_calculos", "lp_confuso", "lp_funcionou_bem",
                     "lp_problema_tecnico", "lp_nao_usou", "lp_outro"]].sum())

lp_deu_codigo           1.0
lp_fez_calculos         1.0
lp_confuso              8.0
lp_funcionou_bem       68.0
lp_problema_tecnico     8.0
lp_nao_usou             8.0
lp_outro                1.0
dtype: float64


### 2.3. Identificação de duplicados para decisão posterior

In [37]:
#Sinalizar duplicados para decisão caso a caso antes do merge com dados pre
pos_duplicados = df_pos[df_pos["participant_id"].duplicated(keep=False)]
pos_duplicados = pos_duplicados.sort_values(["participant_id", "start_time"])
pos_duplicados[["participant_id", "start_time", "end_time"]]

,participant_id,start_time,end_time
37,113,2026-09-21 17:14:09,2026-09-21 17:17:29
65,113,2026-09-21 18:19:17,2026-09-21 18:31:15


## 3. Follow-UP

### 3.1. Importar Follow-UP

In [38]:
df_follow_up = pd.read_excel(PASTA / "followup.xlsx")
df_follow_up

,ID,Hora de início,Hora de conclusão,E-mail,Nome,Hora da última modificação,Código de participante,"Precisas de preparar 250 mL de uma solução de cloreto de potássio (KCl), com concentração 30 mM. Sabendo que a massa molar do KCl é 74,55 g/mol, que massa (em mg) deves pesar?",Qual a estratégia de resolução que utilizaste?,"Uma solução concentrada de gluconato de cálcio a 1,8 M precisa de ser diluída até 0,45 M antes da administração. Partindo de 12 mL da solução concentrada, que volume de soro deves adicionar?",Qual a estratégia de resolução que utilizaste?2,"Para preparar uma curva padrão de doseamento de um fármaco, fizeste 3 diluições seriadas sucessivas, cada uma com fator 1:10. A concentração final da última diluição foi de 5 µg/mL. Qual era a con...",Qual a estratégia de resolução que utilizaste?3,Voltaste a praticar/estudar os conceitos desde a aula?,Consideraste a sessão DILU-IA útil?,Que dispositivo utilizaste durante a sessão do DILU-IA?,"Durante a sessão, conseguiste aceder ao LabPartner?\n",Porque não usaste o LabPartner?,Obrigada pela participação!
0,3,2026-10-10 10:47:01,2026-10-10 10:52:02,anonymous,NaN,NaN,P074,559.000,Fórmula matemática,36.000,Fórmula matemática,135,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",4,Computador;,"Sim, acedi e usei-o",NaN,"Plataforma intuitiva, apenas tendo de melhorar..."
1,4,2026-10-10 10:52:58,2026-10-10 11:06:56,anonymous,NaN,NaN,P073,561.000,Fórmula matemática,48.000,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com os exercícios da aula.",4,Computador;,"Sim, acedi e usei-o",NaN,A dificuldade em usar o LabPartner surgiu porq...
2,5,2026-10-10 11:35:19,2026-10-10 11:38:17,anonymous,NaN,NaN,P048,590.000,Fórmula matemática,36.000,Regra de três simples,5000,"Resolvi por etapas, calculando cada diluição e...",Não.,3,Tablet;,"Sim, acedi e usei-o",NaN,.
3,6,2026-10-10 15:27:08,2026-10-10 15:36:20,anonymous,NaN,NaN,P090,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",3,Computador;,"Sim, acedi e usei-o",NaN,"O LabPartner poderá ser muito útil, no entanto..."
4,7,2026-10-10 15:27:08,2026-10-10 15:37:24,anonymous,NaN,NaN,P125,559.100,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",3,Computador;,"Sim, acedi e usei-o",NaN,.
5,8,2026-10-10 17:32:49,2026-10-10 17:37:24,anonymous,NaN,NaN,P091,559125.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",3,Computador;,"Sim, acedi, mas quase não o usei",Preferi pedir ajuda aos docentes ou a colegas,Penso que o LabPartner é uma mecanismo de apre...
6,9,2026-10-10 19:47:53,2026-10-10 19:58:53,anonymous,NaN,NaN,P147,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",5,Telemóvel;,"Tentei, mas não consegui aceder (por exemplo, ...",NaN,Gostei!
7,10,2026-10-11 16:13:24,2026-10-11 16:18:48,anonymous,NaN,NaN,P016,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com outros recursos.",4,Telemóvel;,"Sim, acedi e usei-o",NaN,Achei a DILU-IA uma ferramenta bastante intera...
8,11,2026-10-11 17:14:05,2026-10-11 17:20:44,anonymous,NaN,NaN,P097,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...",Não.,3,Telemóvel;,"Tentei, mas não consegui aceder (por exemplo, ...",NaN,Acho que não obteve o efeito desejado já que n...
9,12,2026-10-11 18:15:32,2026-10-11 18:24:42,anonymous,NaN,NaN,P132,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com os exercícios da aula.",4,Tablet;Computador;,"Sim, acedi, mas quase não o usei",Preferi pedir ajuda aos docentes ou a co

In [39]:
print(df_follow_up.shape)
df_follow_up.columns.tolist()

(49, 19)


['ID',
 'Hora de início',
 'Hora de conclusão',
 'E-mail',
 'Nome',
 'Hora da última modificação',
 'Código de participante',
 'Precisas de preparar 250 mL de uma solução de cloreto de potássio (KCl), com concentração 30 mM. Sabendo que a massa molar do KCl é 74,55 g/mol, que massa (em mg) deves pesar?',
 'Qual a estratégia de resolução que utilizaste?',
 'Uma solução concentrada de gluconato de cálcio a 1,8 M precisa de ser diluída até 0,45 M antes da administração. Partindo de 12 mL da solução concentrada, que volume de soro deves adicionar?',
 'Qual a estratégia de resolução que utilizaste?2',
 'Para preparar uma curva padrão de doseamento de um fármaco, fizeste 3 diluições seriadas sucessivas, cada uma com fator 1:10. A concentração final da última diluição foi de 5 µg/mL. Qual era a con...',
 'Qual a estratégia de resolução que utilizaste?3',
 'Voltaste a praticar/estudar os conceitos desde a aula?',
 'Consideraste a sessão DILU-IA útil?',
 'Que dispositivo utilizaste durante a se

### 3.2. Limpeza inicial follow-UP

1. Removidas colunas administrativas
2. Colunas renomeadas
5. Uniformização codigos de participante
6. Cálculo do tempo de preenchimento do questionário
8. Identificação do numero de submissões para o fluxograma

In [40]:
# Remover colunas
df_follow_up.drop(columns=['ID', 'E-mail', 'Nome', 'Hora da última modificação'],
            inplace=True, errors='ignore')

In [41]:
#Renomear colunas

new_names_follow_up = {
    'Hora de início' : 'start_time',
    'Hora de conclusão' : 'end_time',
    'Código de participante' : 'participant_id',
    'Precisas de preparar 250 mL de uma solução de cloreto de potássio (KCl), com concentração 30 mM. Sabendo que a massa molar do KCl é 74,55 g/mol, que massa (em mg) deves pesar?' : 'ex1_answer',
    'Qual a estratégia de resolução que utilizaste?' : 'ex1_strategy',
    'Uma solução concentrada de gluconato de cálcio a 1,8 M precisa de ser diluída até 0,45 M antes da administração. Partindo de 12 mL da solução concentrada, que volume de soro deves adicionar?' : 'ex2_answer',
    'Qual a estratégia de resolução que utilizaste?2' : 'ex2_strategy',
    'Para preparar uma curva padrão de doseamento de um fármaco, fizeste 3 diluições seriadas sucessivas, cada uma com fator 1:10. A concentração final da última diluição foi de 5 µg/mL. Qual era a con...' : 'ex3_answer',
    'Qual a estratégia de resolução que utilizaste?3' : 'ex3_strategy',
    #demograficas
     'Voltaste a praticar/estudar os conceitos desde a aula?' : 'more_practice',
    'Consideraste a sessão DILU-IA útil?' : 'perceived_usefulness',
    'Que dispositivo utilizaste durante a sessão do DILU-IA?' : 'tech_used',
    'Durante a sessão, conseguiste aceder ao LabPartner?\n' : 'ai_use_raw',
    'Porque não usaste o LabPartner?' : 'reason_not_ai_use',
    'Obrigada pela participação!' : 'comments'
}

df_follow_up = df_follow_up.rename(columns=new_names_follow_up, errors='ignore')
print("Colunas renomeadas.")

Colunas renomeadas.


In [42]:
df_follow_up["participant_id_raw"] = df_follow_up["participant_id"].astype(str)
codigo = df_follow_up["participant_id_raw"].str.extract(r"(\d+)", expand=False)

# Ver primeiro os casos problemáticos
print(df_follow_up.loc[codigo.isna(), ["participant_id_raw", "start_time"]])
estranhos = ~df_follow_up["participant_id_raw"].str.strip().str.fullmatch(r"[Pp]?\d+")
print(df_follow_up.loc[estranhos, ["participant_id_raw", "start_time"]])

Empty DataFrame
Columns: [participant_id_raw, start_time]
Index: []
   participant_id_raw          start_time
32               PO12 2026-10-13 14:10:20


In [43]:
# Normalizar participant_id (uppercase, remover espaços)
df_follow_up['participant_id'] = (df_follow_up['participant_id'].astype(str).str.extract(r'(\d+)', expand = False).astype(int))
df_follow_up.sort_values('participant_id', inplace=True)

# Calcular tempo de preenchimento
df_follow_up['fill_in_time'] = ((df_follow_up['end_time'] - df_follow_up['start_time'])
                                   .dt.total_seconds() / 60).round(2)

print(f"Dataset pré processado: {df_follow_up.shape}")
df_follow_up.head()

Dataset pré processado: (49, 17)


,start_time,end_time,participant_id,ex1_answer,ex1_strategy,ex2_answer,ex2_strategy,ex3_answer,ex3_strategy,more_practice,perceived_usefulness,tech_used,ai_use_raw,reason_not_ai_use,comments,participant_id_raw,fill_in_time
24,2026-10-12 21:40:09,2026-10-12 23:03:46,5,559.0,Fórmula matemática,36.0,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com a plataforma DILU-IA.",5,Telemóvel;Computador;,"Sim, acedi e usei-o",NaN,Gostei muito e ajudou-me a perceber melhor as ...,P005,83.62
14,2026-10-12 14:12:28,2026-10-12 14:16:24,7,559.0,Fórmula matemática,67500.0,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com os exercícios da aula.",4,Tablet;,"Sim, acedi e usei-o",NaN,Gostei,P007,3.93
32,2026-10-13 14:10:20,2026-10-13 14:14:34,12,559.0,Fórmula matemática,36.0,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,Não.,2,Tablet;,"Sim, acedi, mas quase não o usei",Achei que conseguia resolver sozinho/a,.,PO12,4.23
7,2026-10-11 16:13:24,2026-10-11 16:18:48,16,559.0,Fórmula matemática,36.0,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com outros recursos.",4,Telemóvel;,"Sim, acedi e usei-o",NaN,Achei a DILU-IA uma ferramenta bastante intera...,P016,5.40
43,2026-10-13 15:25:27,2026-10-13 15:27:01,16,559.0,Fórmula matemática,36.0,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",4,Computador;,"Sim, acedi e usei-o",NaN,Achei diferente!,P016,1.57


In [44]:
SESSAO = pd.Timestamp("2026-09-21")

df_follow_up["dias_desde_sessao"] = (df_follow_up["end_time"].dt.normalize() - SESSAO).dt.days
print(df_follow_up["dias_desde_sessao"].describe())
print(df_follow_up["end_time"].dt.date.value_counts().sort_index())

count    49.000000
mean     21.102041
std       1.065460
min      19.000000
25%      21.000000
50%      21.000000
75%      22.000000
max      22.000000
Name: dias_desde_sessao, dtype: float64
end_time
2026-10-10     7
2026-10-11     4
2026-10-12    15
2026-10-13    23
Name: count, dtype: int64


In [45]:
  dia13 = df_follow_up[df_follow_up["end_time"].dt.date == pd.Timestamp("2026-10-13").date()]
  print(dia13["end_time"].dt.floor("30min").value_counts().sort_index())

end_time
2026-10-13 09:30:00    2
2026-10-13 12:30:00    4
2026-10-13 14:00:00    2
2026-10-13 14:30:00    5
2026-10-13 15:00:00    7
2026-10-13 15:30:00    2
2026-10-13 16:30:00    1
Name: count, dtype: int64


In [46]:
for col in ["ai_use_raw", "reason_not_ai_use", "more_practice", "tech_used", "perceived_usefulness"]:
    print(f"\n--- {col}")
    print(df_follow_up[col].value_counts(dropna=False))


--- ai_use_raw
ai_use_raw
Sim, acedi e usei-o                                                 18
Tentei, mas não consegui aceder (por exemplo, problema de login)    15
Sim, acedi, mas quase não o usei                                     9
Não tentei aceder                                                    4
Não me lembro                                                        3
Name: count, dtype: int64

--- reason_not_ai_use
reason_not_ai_use
NaN                                              36
Preferi pedir ajuda aos docentes ou a colegas     5
Achei que conseguia resolver sozinho/a            4
Ele não respondia às dúvidas de forma direta.     1
Dava pouca informação                             1
Não tive tempo                                    1
Não consegui aceder                               1
Name: count, dtype: int64

--- more_practice
more_practice
Sim, com os exercícios da aula.    19
Não.                               17
Sim, com outros recursos.          11
Sim, com a pla

In [47]:
#Codificar variáveis de uso

ACESSO = {
    "Sim, acedi e usei-o": 1,
    "Sim, acedi, mas quase não o usei": 1,
    "Tentei, mas não consegui aceder (por exemplo, problema de login)": 0,
    "Não tentei aceder": np.nan,
    "Não me lembro": np.nan,
}
USO = {
    "Sim, acedi e usei-o": 2,
    "Sim, acedi, mas quase não o usei": 1,
    "Tentei, mas não consegui aceder (por exemplo, problema de login)": 0,
    "Não tentei aceder": 0,
    "Não me lembro": np.nan,
}

respostas = df_follow_up["ai_use_raw"].str.strip()
assert respostas.dropna().isin(ACESSO.keys()).all(), "Há respostas que não estão nos dicionários"

df_follow_up["ai_access"] = respostas.map(ACESSO)
df_follow_up["ai_use"] = respostas.map(USO)

In [48]:
# Codificar “Porque não usaste o LabPartner?
MOTIVOS = {
    "Achei que conseguia resolver sozinho/a": "motivo_sozinho",
    "Não consegui aceder": "motivo_sem_acesso",
    "Não tive tempo": "motivo_sem_tempo",
    "Preferi pedir ajuda aos docentes ou a colegas": "motivo_ajuda_humana",
    "Não percebi como funcionava": "motivo_nao_percebi",
}
texto_motivo = df_follow_up["reason_not_ai_use"].fillna("")
for opcao, coluna in MOTIVOS.items():
    df_follow_up[coluna] = texto_motivo.str.contains(opcao, regex=False).astype(float)

# Partes que não correspondem a nenhuma opção (texto de "Outro" ou opção escrita de outra forma)
def motivo_outro(texto):
    if pd.isna(texto):
        return np.nan
    partes = [p.strip() for p in str(texto).split(";") if p.strip()]
    outras = [p for p in partes if p not in MOTIVOS]
    return "; ".join(outras) if outras else np.nan

df_follow_up["motivo_outro_texto"] = df_follow_up["reason_not_ai_use"].apply(motivo_outro)
colunas_motivo = list(MOTIVOS.values())
df_follow_up.loc[df_follow_up["reason_not_ai_use"].isna(), colunas_motivo] = np.nan
print(df_follow_up["motivo_outro_texto"].dropna().value_counts())

motivo_outro_texto
Ele não respondia às dúvidas de forma direta.    1
Dava pouca informação                            1
Name: count, dtype: int64


In [49]:
   df_follow_up["motivo_outro"] = df_follow_up["motivo_outro_texto"].notna().astype(float)
   df_follow_up.loc[df_follow_up["reason_not_ai_use"].isna(), "motivo_outro"] = np.nan

In [50]:
incoerente = (df_follow_up["ai_use_raw"].str.strip() == "Sim, acedi e usei-o") \
             & df_follow_up["reason_not_ai_use"].notna()
print("Usou e deu motivo para não usar:", incoerente.sum())
df_follow_up.loc[incoerente, ["participant_id", "ai_use_raw", "reason_not_ai_use"]]

Usou e deu motivo para não usar: 0


,participant_id,ai_use_raw,reason_not_ai_use


In [51]:
df_follow_up

,start_time,end_time,participant_id,ex1_answer,ex1_strategy,ex2_answer,ex2_strategy,ex3_answer,ex3_strategy,more_practice,...,dias_desde_sessao,ai_access,ai_use,motivo_sozinho,motivo_sem_acesso,motivo_sem_tempo,motivo_ajuda_humana,motivo_nao_percebi,motivo_outro_texto,motivo_outro
24,2026-10-12 21:40:09,2026-10-12 23:03:46,5,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com a plataforma DILU-IA.",...,21,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,2026-10-12 14:12:28,2026-10-12 14:16:24,7,559.000,Fórmula matemática,67500.000,Fórmula matemática,5000,"Resolvi por etapas, calculando cada diluição e...","Sim, com os exercícios da aula.",...,21,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,2026-10-13 14:10:20,2026-10-13 14:14:34,12,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,Não.,...,22,1.0,1.0,1.0,0.0,0.0,0.0,0.0,NaN,0.0
7,2026-10-11 16:13:24,2026-10-11 16:18:48,16,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com outros recursos.",...,20,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,2026-10-13 15:25:27,2026-10-13 15:27:01,16,559.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",...,22,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,2026-10-12 12:47:16,2026-10-12 12:52:33,19,559.000,Regra de três simples,36.000,Regra de três simples,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com outros recursos.",...,21,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2026-10-13 09:33:17,2026-10-13 09:40:10,34,522.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,"Sim, com os exercícios da aula.",...,22,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,2026-10-12 22:16:21,2026-10-12 22:24:40,35,559.000,Fórmula matemática,35.000,Fórmula matemática,5000,Desenhei um esquema,"Sim, com os exercícios da aula.",...,21,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2026-10-13 09:30:10,2026-10-13 09:40:02,39,522.000,Fórmula matemática,36.000,Fórmula matemática,5000,Fórmula matemática (aplicada diretamente ao re...,Não.,...,22,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,2026-10-13 15:09:55,2026-10-13 15:39:59,40,560.000,Fórmula matemática,36.000,Fórmula matemática,5000,Regra de três simples,"Sim, com os exercícios da aula.",...,22,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
#Caracterizar tipo de pratica adicional
PRATICA = {
    "Sim, com os exercícios da aula.": "exercicios_aula",
    "Sim, com a plataforma DILU-IA.": "plataforma",
    "Sim, com outros recursos.": "outros_recursos",
    "Não.": "nao",
}
resposta = df_follow_up["more_practice"].str.strip()
df_follow_up["practice_source"] = resposta.map(PRATICA)

# respostas "Outro": texto livre, não está no dicionário
outros = resposta.notna() & df_follow_up["practice_source"].isna()
df_follow_up.loc[outros, ["more_practice"]]

,more_practice


In [53]:
df_follow_up["practiced"] = (df_follow_up["practice_source"] != "nao").astype(float)
df_follow_up.loc[df_follow_up["practice_source"].isna(), "practiced"] = np.nan

In [54]:
#Carcaterizar dispositivos disponiveis
dispositivos = df_follow_up["tech_used"].fillna("")

df_follow_up["tech_phone"] = dispositivos.str.contains("Telemóvel").astype(float)
df_follow_up["tech_tablet"] = dispositivos.str.contains("Tablet").astype(float)
df_follow_up["tech_computer"] = dispositivos.str.contains("Computador").astype(float)
df_follow_up["tech_dont_remember"] = dispositivos.str.contains("Não me lembro").astype(float)
df_follow_up["tech_n"] = df_follow_up[["tech_phone", "tech_tablet", "tech_computer"]].sum(axis=1)

colunas_tech = ["tech_phone", "tech_tablet", "tech_computer", "tech_dont_remember", "tech_n"]
df_follow_up.loc[df_follow_up["tech_used"].isna(), colunas_tech] = np.nan

In [55]:
fluxogram["follow_up_submissions"] = len(df_follow_up)

In [56]:
fluxogram

{'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49}

### 3.3. Identificação de duplicados para decisão posterior

In [57]:
#Sinalizar duplicados para decisão caso a caso antes do merge com dados pre
follow_duplicados = df_follow_up[df_follow_up["participant_id"].duplicated(keep=False)]
follow_duplicados = follow_duplicados.sort_values(["participant_id", "start_time"])
follow_duplicados[["participant_id", "start_time", "end_time"]]

,participant_id,start_time,end_time
7,16,2026-10-11 16:13:24,2026-10-11 16:18:48
43,16,2026-10-13 15:25:27,2026-10-13 15:27:01
10,66,2026-10-11 22:36:30,2026-10-11 22:39:11
13,66,2026-10-12 13:18:36,2026-10-12 13:20:49
28,67,2026-10-13 12:48:35,2026-10-13 12:53:11
29,67,2026-10-13 12:48:40,2026-10-13 12:53:16
30,67,2026-10-13 12:48:45,2026-10-13 12:53:22
16,113,2026-10-12 15:59:25,2026-10-12 16:03:26
31,113,2026-10-13 12:57:19,2026-10-13 12:59:40
6,147,2026-10-10 19:47:53,2026-10-10 19:58:53


## 4. Resolução de discrepâncias


### 4.1. Importar lista de inscritos

In [58]:
df_inscritos = pd.read_excel(PASTA / "inscritos_sem_nomes.xlsx")
df_inscritos

,Código de participante,Hora de início,Hora de conclusão
0,P011,2026-09-21 14:40:03,2026-09-21 14:40:43
1,P014,2026-09-21 14:40:12,2026-09-21 14:41:01
2,P015,2026-09-21 14:41:04,2026-09-21 14:42:18
3,P008,2026-09-21 14:41:08,2026-09-21 14:42:34
4,P012,2026-09-21 14:41:07,2026-09-21 14:42:45
...,...,...,...
113,P146,NaT,NaT
114,P050,NaT,NaT
115,P147,NaT,NaT
116,P028,NaT,NaT


In [59]:
df_inscritos.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['Código de participante', 'Hora de início', 'Hora de conclusão'], dtype='str')>

In [60]:
#Renomear colunas

new_names_inscritos = {
    'Hora de início' : 'start_time',
    'Hora de conclusão' : 'end_time',
    'Código de participante' : 'participant_id',
}

df_inscritos = df_inscritos.rename(columns=new_names_inscritos, errors='ignore')
print("Colunas renomeadas.")

Colunas renomeadas.


In [61]:
df_inscritos["participant_id"] = df_inscritos["participant_id"].astype(str).str.extract(r"(\d+)")[0].astype(int)

In [62]:
df_inscritos

,participant_id,start_time,end_time
0,11,2026-09-21 14:40:03,2026-09-21 14:40:43
1,14,2026-09-21 14:40:12,2026-09-21 14:41:01
2,15,2026-09-21 14:41:04,2026-09-21 14:42:18
3,8,2026-09-21 14:41:08,2026-09-21 14:42:34
4,12,2026-09-21 14:41:07,2026-09-21 14:42:45
...,...,...,...
113,146,NaT,NaT
114,50,NaT,NaT
115,147,NaT,NaT
116,28,NaT,NaT


### 4.2. Atribuir turnos

In [63]:
df_inscritos["start_time"] = pd.to_datetime(df_inscritos["start_time"])
df_pre["start_time"] = pd.to_datetime(df_pre["start_time"])
df_pos["start_time"] = pd.to_datetime(df_pos["start_time"])

# PREENCHER: início e fim de cada turno, sem intervalos entre eles
TURNOS = {
    1: ("2026-09-21 14:30", "2026-09-21 16:00"),
    2: ("2026-09-21 16:00", "2026-09-21 17:30"),
    3: ("2026-09-21 17:30", "2026-09-21 19:00"),
}

def atribuir_turno(hora):
    if pd.isna(hora):
        return np.nan
    for turno, (inicio, fim) in TURNOS.items():
        if pd.Timestamp(inicio) <= hora < pd.Timestamp(fim):
            return turno
    return np.nan

df_inscritos["turno"] = df_inscritos["start_time"].apply(atribuir_turno)
df_pre["turno"] = df_pre["start_time"].apply(atribuir_turno)
df_pos["turno"] = df_pos["start_time"].apply(atribuir_turno)

In [64]:
##Verificar pessoas sem turno
print(df_pre["turno"].isna().sum(), df_pos["turno"].isna().sum())
print(df_inscritos["turno"].isna().sum())

0 1
8


In [65]:
#Identificar Discrepancia
turnos = df_pre[["participant_id", "turno"]].merge(
    df_pos[["participant_id", "turno"]], on="participant_id", suffixes=("_pre", "_pos"))
turnos[turnos["turno_pre"] != turnos["turno_pos"]]

,participant_id,turno_pre,turno_pos
56,89,3,NaN
71,113,2,3.0
72,113,3,2.0


In [66]:
df_pos[df_pos["participant_id"] == 89][["participant_id", "start_time", "end_time", "game_finished"]]

,participant_id,start_time,end_time,game_finished
92,89,2026-09-21 21:01:58,2026-09-21 21:17:50,1.0


**Decisão**

89 pertencia ao 3ºturno, deverá ter ido para casa e no proprio dia terminou a submissão, provavelmente por falta de tempo na aula
Troco manualmente o turno do pos para o turno 3. Deixo nota por resposta fora de aula.

In [67]:
df_pos.loc[df_pos["participant_id"] == 89, "turno"] = 3

In [68]:
df_pos["pos_fora_sessao"] = 0
MARGEM = pd.Timedelta(minutes=15)   # DECISÃO: tolerância depois do fim do turno
fim_do_turno = df_pos["turno"].map({t: pd.Timestamp(fim) for t, (inicio, fim) in TURNOS.items()})
df_pos["pos_fora_sessao"] = (df_pos["end_time"] > fim_do_turno + MARGEM).astype(int)

df_pos.loc[df_pos["pos_fora_sessao"] == 1, ["participant_id", "turno", "start_time", "end_time"]]

,participant_id,turno,start_time,end_time
91,5,1.0,2026-09-21 15:52:33,2026-09-21 20:32:28
92,89,3.0,2026-09-21 21:01:58,2026-09-21 21:17:50


In [69]:
fluxogram["pos_fora_sessao"] = int(df_pos["pos_fora_sessao"].sum())

In [70]:
fluxogram

{'pre_submissions': 119,
 'pos_submissions': 93,
 'follow_up_submissions': 49,
 'pos_fora_sessao': 2}

### 4.3. Listar discrepancias entre inscritos e submissões

In [71]:
inscritos = set(df_inscritos["participant_id"])

pre_nao_inscritos = df_pre[~df_pre["participant_id"].isin(inscritos)]
pos_nao_inscritos = df_pos[~df_pos["participant_id"].isin(inscritos)]
inscritos_sem_dados = df_inscritos[~df_inscritos["participant_id"].isin(df_pre["participant_id"])
                                   & ~df_inscritos["participant_id"].isin(df_pos["participant_id"])]


In [72]:
pre_nao_inscritos[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
14,29,2026-09-21 14:50:17,1
18,30,2026-09-21 14:50:17,1
19,31,2026-09-21 14:51:28,1
10,33,2026-09-21 14:50:03,1
82,54,2026-09-21 17:37:10,3


In [73]:
pos_nao_inscritos[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
12,29,2026-09-21 15:45:14,1.0
2,30,2026-09-21 15:44:02,1.0
4,31,2026-09-21 15:42:48,1.0
6,33,2026-09-21 15:43:36,1.0


### 4.4. Resolver duplicados e discrepâncias

Encontramos 3 ids com submissão duplicada:
- No pré:
    - 10: Submeteu dois questionários pré, um (com mais tempo em questionário) e depois o outro (com menos tempo de preenchimento);
        - **Decisão:** assume-se o 1º formulário
    - 59: Duas submissões simultaneas com o mesmo código;
        - Hipoteses: mesma pessoa com dois separadores abertos ou duas pessoas a trabalhar juntas;
        - **Próximo passo:** Verificar formularios sem par no pre e pos
- No pré e no pós:
    - 113: Duas submisões com o mesmo codigo, em turnos diferentes.
        - **Próximo passo:** verificar inscrições para tentar identificar os códigos reais.

In [74]:
excluidos = []   # tudo o que sai fica aqui, com o motivo, para o fluxograma

#### 4.4.1. Participante 10

In [75]:
df_pre[df_pre["participant_id"] == 10][["participant_id", "start_time", "end_time"]]

,participant_id,start_time,end_time
78,10,2026-09-21 16:45:47,2026-09-21 16:53:59
73,10,2026-09-21 16:17:45,2026-09-21 16:42:27


In [76]:
excluidos.append(df_pre.loc[[78]].assign(motivo="ressubmissão: fica a primeira"))
df_pre = df_pre.drop(index=78)

In [77]:
df_pre[df_pre["participant_id"] == 10][["participant_id", "start_time", "end_time"]]

,participant_id,start_time,end_time
73,10,2026-09-21 16:17:45,2026-09-21 16:42:27


#### 4.4.2. Participantes 59 

Contexto: Há duas submissões simultaneas com o nr 59 no pré. Procuramos se ha submissões duplicadas com o numero 59 e alguma submissão no pos que nao tenha submissão no pré

In [78]:
df_pos[df_pos["participant_id"] == 59][["participant_id", "start_time", "end_time", "turno"]]

,participant_id,start_time,end_time,turno
0,59,2026-09-21 15:42:58,2026-09-21 15:46:00,1.0


In [79]:
pos_sem_par = df_pos[~df_pos["participant_id"].isin(df_pre["participant_id"])]
pos_sem_par.sort_values("start_time")[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
3,58,2026-09-21 15:46:36,1.0


Racional:

Provavelmente os dois 59 no pré são o 58 e 59 a submeterem ao mesmo tempo e provavelmente a trabalharem lado a lado (tempos de submissão contiguos)

**Decisão:**
- Manter dados.
- Atribuir codigos não emparalháveis por impossibilidade de saber quais dos 59 é o 58.
- Mantem se submisssões sozinhas para analise descritiva geral. 

In [80]:
df_pre[df_pre["participant_id"] == 59][["participant_id", "start_time"]]

,participant_id,start_time
4,59,2026-09-21 14:48:42
2,59,2026-09-21 14:49:22


In [81]:
assert not df_pre["participant_id"].isin([591, 592]).any()

df_pre.loc[4, "participant_id"] = 591
df_pre.loc[2, "participant_id"] = 592

df_pre["pre_pos_nao_emparelhavel"] = 0
df_pre.loc[df_pre["participant_id"].isin([591, 592]), "pre_pos_nao_emparelhavel"] = 1

df_pos["pre_pos_nao_emparelhavel"] = 0
df_pos.loc[df_pos["participant_id"].isin([58, 59]), "pre_pos_nao_emparelhavel"] = 1

df_pre[df_pre["participant_id"].isin([591, 592])][["participant_id", "start_time", "pre_pos_nao_emparelhavel"]]

#### 4.4.3. Participantes 113

In [82]:
df_pre[df_pre["participant_id"] == 113][["participant_id", "start_time", "gender", "education", "digital_confort", "ia_confort", "turno"]]

,participant_id,start_time,gender,education,digital_confort,ia_confort,turno
51,113,2026-09-21 16:15:37,0,Ensino Secundário,4,3,2
83,113,2026-09-21 17:41:23,1,Ensino Secundário,4,4,3


In [83]:
inscritos_sem_dados[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
28,40,2026-09-21 14:46:53,1.0
33,18,2026-09-21 14:46:42,1.0
73,98,2026-09-21 16:20:49,2.0
86,133,2026-09-21 17:37:19,3.0
110,6,NaT,NaN
114,50,NaT,NaN


In [84]:
#verificando se 113 está inscrito no turno 2

In [85]:
df_inscritos[df_inscritos["participant_id"].isin([113, 133])][["participant_id","turno"]]

,participant_id,turno
36,113,2.0
86,133,3.0


Racional:

Tenho submissões duplicadas no pre e no pos com o numero 113 em turnos diferenes 2 e 3.
Tenho 1 inscrito com o nr 113 no turno 2.
Tenho 1 inscrito com o nr 133 no turno 3, sem submissões.

**Decisão:**
Dada a similaridade dos códigos, dou manualmente o número 133 ao 113 do turno 3.

In [86]:
assert not df_pre["participant_id"].isin([133]).any()

df_pre.loc[(df_pre["participant_id"] == 113) & (df_pre["turno"] == 3), "participant_id"] = 133
df_pos.loc[(df_pos["participant_id"] == 113) & (df_pos["turno"] == 3), "participant_id"] = 133

#### 4.4.4. Atualizar registo de decisões

In [87]:
assert not df_pre["participant_id"].duplicated().any()
assert not df_pos["participant_id"].duplicated().any()

fluxogram["pre_ressubmissoes_removidas"] = 1
fluxogram["codigos_reconciliados"] = 1
fluxogram["nao_emparelhaveis"] = 2
print(fluxogram)

{'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49, 'pos_fora_sessao': 2, 'pre_ressubmissoes_removidas': 1, 'codigos_reconciliados': 1, 'nao_emparelhaveis': 2}


#### 4.4.5. Follow-Up

Encontramos os seguintes duplicados no follow-up:

 - 16, 66, 147: submissões da mesma pessoa, em dias diferentes.

    **Decisão:** Provável ressubmissão, fica a primeira.

 - 67: três submissões com 5 segundos de intervalo entre si, com a mesma duração (cerca de 4 min 36 s)

   **Interpretação**: pessoas a responder juntas com o mesmo código à semelhança do 58 e 59.

   
 - 113: duas submissões, a 12/10 e a 13/10.

   **Interpretação:** Pode ser a confusão 133 e 113 novamente.


   Abordagem manual: verificar respostas 113 e 67


In [88]:
# Decisão: ressubmissões da mesma pessoa (dias diferentes) → fica a primeira
for pid in [16, 66, 147]:
    linhas = df_follow_up[df_follow_up["participant_id"] == pid].sort_values("start_time")
    a_sair = linhas.index[1:]
    excluidos.append(df_follow_up.loc[a_sair].assign(motivo="follow-up: ressubmissão, fica a primeira"))
    df_follow_up = df_follow_up.drop(index=a_sair)

fluxogram["follow_ressubmissoes_removidas"] = 3

In [89]:
COLS_VER = ["participant_id", "start_time", "ex1_answer", "ex2_answer", "ex3_answer",
            "ex1_strategy", "ex2_strategy", "ex3_strategy", "ai_use_raw", "tech_used"]
display(df_follow_up[df_follow_up["participant_id"].isin([67, 113])][COLS_VER])

,participant_id,start_time,ex1_answer,ex2_answer,ex3_answer,ex1_strategy,ex2_strategy,ex3_strategy,ai_use_raw,tech_used
29,67,2026-10-13 12:48:40,559.0,36.0,5000,Fórmula matemática,Fórmula matemática,"Resolvi por etapas, calculando cada diluição e...","Sim, acedi e usei-o",Tablet;
28,67,2026-10-13 12:48:35,559.0,36.0,5000,Fórmula matemática,Fórmula matemática,"Resolvi por etapas, calculando cada diluição e...","Sim, acedi e usei-o",Tablet;
30,67,2026-10-13 12:48:45,559.0,36.0,5000,Fórmula matemática,Fórmula matemática,"Resolvi por etapas, calculando cada diluição e...","Sim, acedi e usei-o",Tablet;
16,113,2026-10-12 15:59:25,559.0,36.0,5000,Fórmula matemática,Fórmula matemática,Regra de três simples,"Sim, acedi e usei-o",Telemóvel;
31,113,2026-10-13 12:57:19,559.0,36.0,500,Fórmula matemática,Fórmula matemática,"Resolvi por etapas, calculando cada diluição e...","Tentei, mas não consegui aceder (por exemplo, ...",Telemóvel;


In [90]:
# 67: três submissões idênticas com 5 s de intervalo → fica a primeira
# Provável submissão repetida por falha de ligação (respostas, método, acesso e dispositivo iguais)
linhas = df_follow_up[df_follow_up["participant_id"] == 67].sort_values("start_time")
excluidos.append(df_follow_up.loc[linhas.index[1:]]
                 .assign(motivo="follow-up: submissão repetida idêntica (provável falha de ligação)"))
df_follow_up = df_follow_up.drop(index=linhas.index[1:])

In [91]:
# 113: duas submissões diferentes (provavelmente 113 e 133) → não atribuíveis
assert not df_follow_up["participant_id"].isin([1131, 1132]).any()
idx = df_follow_up[df_follow_up["participant_id"] == 113].sort_values("start_time").index
df_follow_up.loc[idx[0], "participant_id"] = 1131
df_follow_up.loc[idx[1], "participant_id"] = 1132

df_follow_up["follow_nao_atribuivel"] = df_follow_up["participant_id"].isin([1131, 1132]).astype(int)

assert not df_follow_up["participant_id"].duplicated().any()
fluxogram["follow_submissoes_repetidas_removidas"] = 2
fluxogram["follow_nao_atribuiveis"] = 2
print(len(df_follow_up), fluxogram)

44 {'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49, 'pos_fora_sessao': 2, 'pre_ressubmissoes_removidas': 1, 'codigos_reconciliados': 1, 'nao_emparelhaveis': 2, 'follow_ressubmissoes_removidas': 3, 'follow_submissoes_repetidas_removidas': 2, 'follow_nao_atribuiveis': 2}


### 4.5. Excluir quem não tem consentimento 

In [92]:
inscritos = set(df_inscritos["participant_id"])
inscritos = inscritos | {591, 592} #códigos não emparelhaveis atribuidos manualmente

In [93]:
pre_nao_inscritos = df_pre[~df_pre["participant_id"].isin(inscritos)].copy()
pos_nao_inscritos = df_pos[~df_pos["participant_id"].isin(inscritos)].copy()
print(len(pre_nao_inscritos), len(pos_nao_inscritos))


5 4


In [94]:
pre_nao_inscritos[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
14,29,2026-09-21 14:50:17,1
18,30,2026-09-21 14:50:17,1
19,31,2026-09-21 14:51:28,1
10,33,2026-09-21 14:50:03,1
82,54,2026-09-21 17:37:10,3


In [95]:
pos_nao_inscritos[["participant_id", "start_time", "turno"]]

,participant_id,start_time,turno
12,29,2026-09-21 15:45:14,1.0
2,30,2026-09-21 15:44:02,1.0
4,31,2026-09-21 15:42:48,1.0
6,33,2026-09-21 15:43:36,1.0


In [96]:
inscritos_sem_dados = df_inscritos[~df_inscritos["participant_id"].isin(df_pre["participant_id"])
                                   & ~df_inscritos["participant_id"].isin(df_pos["participant_id"])]
inscritos_sem_dados[["participant_id", "turno"]]

,participant_id,turno
28,40,1.0
33,18,1.0
73,98,2.0
110,6,NaN
114,50,NaN


In [97]:
excluidos.append(pre_nao_inscritos.assign(motivo="sem consentimento registado"))
excluidos.append(pos_nao_inscritos.assign(motivo="sem consentimento registado"))

df_pre = df_pre[df_pre["participant_id"].isin(inscritos)].copy()
df_pos = df_pos[df_pos["participant_id"].isin(inscritos)].copy()

In [98]:
fluxogram["inscritos"] = len(df_inscritos)
fluxogram["inscritos_sem_submissoes"] = len(inscritos_sem_dados)
fluxogram["pre_sem_consentimento"] = len(pre_nao_inscritos)
fluxogram["pos_sem_consentimento"] = len(pos_nao_inscritos)
print(fluxogram)

{'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49, 'pos_fora_sessao': 2, 'pre_ressubmissoes_removidas': 1, 'codigos_reconciliados': 1, 'nao_emparelhaveis': 2, 'follow_ressubmissoes_removidas': 3, 'follow_submissoes_repetidas_removidas': 2, 'follow_nao_atribuiveis': 2, 'inscritos': 118, 'inscritos_sem_submissoes': 5, 'pre_sem_consentimento': 5, 'pos_sem_consentimento': 4}


In [99]:
pre_nao_inscritos = df_pre[~df_pre["participant_id"].isin(inscritos)]
pos_nao_inscritos = df_pos[~df_pos["participant_id"].isin(inscritos)]
print(len(pre_nao_inscritos), len(pos_nao_inscritos))

0 0


In [100]:
#consentimento e códigos do follow-up
inscritos_ids = set(df_inscritos["participant_id"]) | {1131, 1132}

follow_nao_inscritos = df_follow_up[~df_follow_up["participant_id"].isin(inscritos_ids)]
print("Follow-up com código fora dos inscritos:", len(follow_nao_inscritos))
display(follow_nao_inscritos[["participant_id", "participant_id_raw", "end_time"]])

# Códigos com histórico de problemas
display(df_follow_up[df_follow_up["participant_id"].isin([113, 133, 58, 59])]
        [["participant_id", "participant_id_raw", "end_time"]])

Follow-up com código fora dos inscritos: 0


,participant_id,participant_id_raw,end_time


,participant_id,participant_id_raw,end_time


## 5. Ver atrição

In [101]:
pre_valido = df_pre[df_pre["pre_pos_nao_emparelhavel"] == 0]
pos_valido = df_pos[df_pos["pre_pos_nao_emparelhavel"] == 0]

ids_pre = set(pre_valido["participant_id"])
ids_pos = set(pos_valido["participant_id"])

pares = ids_pre & ids_pos
so_pre = ids_pre - ids_pos
so_pos = ids_pos - ids_pre

print("Pares:", len(pares), "| Só pré:", len(so_pre), "| Só pós:", len(so_pos))

Pares: 87 | Só pré: 24 | Só pós: 0


In [102]:
soma = (fluxogram["inscritos_sem_submissoes"] + len(pares) + len(so_pre)
        + len(so_pos) + fluxogram["nao_emparelhaveis"])
print(soma, "vs", fluxogram["inscritos"])
assert soma == fluxogram["inscritos"], "Há inscritos contados duas vezes ou em falta"

118 vs 118


In [103]:
pre_valido = pre_valido.copy()
pre_valido["tem_pos"] = pre_valido["participant_id"].isin(ids_pos)
pd.crosstab(pre_valido["turno"], pre_valido["tem_pos"], margins=True)

tem_pos,False,True,All
turno,,,
1,4,29,33
2,12,28,40
3,8,30,38
All,24,87,111


In [104]:
fluxogram["pares_pre_pos"] = len(pares)
fluxogram["so_pre"] = len(so_pre)
fluxogram["so_pos"] = len(so_pos)
print(fluxogram)

{'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49, 'pos_fora_sessao': 2, 'pre_ressubmissoes_removidas': 1, 'codigos_reconciliados': 1, 'nao_emparelhaveis': 2, 'follow_ressubmissoes_removidas': 3, 'follow_submissoes_repetidas_removidas': 2, 'follow_nao_atribuiveis': 2, 'inscritos': 118, 'inscritos_sem_submissoes': 5, 'pre_sem_consentimento': 5, 'pos_sem_consentimento': 4, 'pares_pre_pos': 87, 'so_pre': 24, 'so_pos': 0}


In [105]:
ids_pre_ok = set(df_pre.loc[df_pre["pre_pos_nao_emparelhavel"] == 0, "participant_id"])  # sem 591/592
ids_pos = set(df_pos["participant_id"])                                          # inclui 58/59 (F9)
ids_follow = set(df_follow_up.loc[df_follow_up["follow_nao_atribuivel"] == 0, "participant_id"])

fluxogram["follow_com_pre"] = len(ids_follow & ids_pre_ok)
fluxogram["follow_com_pos"] = len(ids_follow & ids_pos)
fluxogram["follow_3_momentos"] = len(ids_follow & ids_pre_ok & ids_pos)
fluxogram["follow_sem_pre_nem_pos"] = len(ids_follow - set(df_pre["participant_id"]) - ids_pos)
assert len(ids_follow) == (fluxogram["follow_com_pre"] + fluxogram["follow_sem_pre_nem_pos"]
                           + (fluxogram["follow_com_pos"] - fluxogram["follow_3_momentos"]))
assert len(df_follow_up) == (fluxogram["follow_up_submissions"] - fluxogram["follow_ressubmissoes_removidas"]
                             - fluxogram["follow_submissoes_repetidas_removidas"])
print(fluxogram)

{'pre_submissions': 119, 'pos_submissions': 93, 'follow_up_submissions': 49, 'pos_fora_sessao': 2, 'pre_ressubmissoes_removidas': 1, 'codigos_reconciliados': 1, 'nao_emparelhaveis': 2, 'follow_ressubmissoes_removidas': 3, 'follow_submissoes_repetidas_removidas': 2, 'follow_nao_atribuiveis': 2, 'inscritos': 118, 'inscritos_sem_submissoes': 5, 'pre_sem_consentimento': 5, 'pos_sem_consentimento': 4, 'pares_pre_pos': 87, 'so_pre': 24, 'so_pos': 0, 'follow_com_pre': 41, 'follow_com_pos': 32, 'follow_3_momentos': 32, 'follow_sem_pre_nem_pos': 1}


## 6. Organizar dados para análise

### 6.1. Respostas aos exercícios

#### 6.1.1. Cotar respostas aos exercicios

In [106]:
#empilhar os momentos
COLUNAS_EX = ["participant_id", "ex1_answer", "ex2_answer", "ex3_answer",
              "ex1_strategy", "ex2_strategy", "ex3_strategy"]

pre_ex = df_pre[COLUNAS_EX].copy()
pre_ex["momento"] = "pre"

pos_ex = df_pos[COLUNAS_EX].copy()
pos_ex["momento"] = "pos"

follow_ex = df_follow_up[COLUNAS_EX].copy()
follow_ex["momento"] = "follow"

empilhado = pd.concat([pre_ex, pos_ex, follow_ex], ignore_index=True)

In [107]:
len(empilhado) == len(df_pre) + len(df_pos) + len (df_follow_up)

True

In [108]:
# transformar em 1 linha por exercicios
partes = []
for ex in ["ex1", "ex2", "ex3"]:
    parte = empilhado[["participant_id", "momento", f"{ex}_answer", f"{ex}_strategy"]].copy()
    parte = parte.rename(columns={f"{ex}_answer": "answer_raw", f"{ex}_strategy": "strategy_raw"})
    parte["exercicio"] = ex
    partes.append(parte)

respostas = pd.concat(partes, ignore_index=True)
respostas = respostas.sort_values(["participant_id", "momento", "exercicio"]).reset_index(drop=True)

In [109]:
len(respostas) == 3 * len(empilhado)

True

In [110]:
respostas.dtypes

participant_id      int64
momento               str
answer_raw        float64
strategy_raw          str
exercicio             str
dtype: object

In [111]:
#extrair valores das respostas
respostas["valor"] = pd.to_numeric(respostas["answer_raw"], errors="coerce")
assert (respostas["valor"].isna() == respostas["answer_raw"].isna()).all(), "Algum valor não era número"
respostas["estado"] = np.where(respostas["valor"].isna(), "vazia", "ok")

In [112]:
respostas["estado"].value_counts()

estado
ok    738
Name: count, dtype: int64

In [113]:
CERTAS = {
    "pre":    {"ex1": 451.3875, "ex2": 47.5, "ex3": 1000},
    "pos":    {"ex1": 504.06,   "ex2": 60,   "ex3": 960},
    "follow": {"ex1": 559.125,  "ex2": 36,   "ex3": 5000},
}
VOLUME_FINAL = {"pre": 62.5, "pos": 80, "follow": 48}
TOLERANCIA = 0.5
VAZIA_CONTA_COMO_ERRADA = True

# O enunciado pede uma unidade explícita?
UNIDADE_EXPLICITA = {"ex1": False, "ex2": False, "ex3": True}

In [114]:
def ordem_grandeza(valor, certa):
    """Devolve k se valor = certa × 10^k (dentro da tolerância). Se não, devolve vazio."""
    if pd.isna(valor):
        return np.nan
    for k in [0, 1, -1, 2, -2, 3, -3, 4, -4, 5, -5, 6, -6, 7, -7]:
        if abs(valor / 10**k - certa) <= TOLERANCIA:
            return k
    return np.nan

respostas["ordem"] = respostas.apply(
    lambda linha: ordem_grandeza(linha["valor"], CERTAS[linha["momento"]][linha["exercicio"]]), axis=1)

In [115]:
def classificar(linha):
    if pd.isna(linha["valor"]):
        return "vazia"
    k = linha["ordem"]
    if pd.isna(k):
        return "errado"
    if k == 0:
        return "certo"
    if UNIDADE_EXPLICITA[linha["exercicio"]] and k < 0:
        return "errado_unidade"
    return "rever"

respostas["classificacao"] = respostas.apply(classificar, axis=1)

In [116]:
def erro_volume_final(linha):
    if linha["exercicio"] != "ex2" or pd.isna(linha["valor"]):
        return np.nan
    k = ordem_grandeza(linha["valor"], VOLUME_FINAL[linha["momento"]])
    return 0 if pd.isna(k) else 1

respostas["volume_final"] = respostas.apply(erro_volume_final, axis=1)

In [117]:
para_rever = respostas[(respostas["classificacao"] == "rever")]
para_rever[["participant_id", "momento", "exercicio", "valor", "ordem"]].sort_values(["momento", "exercicio", "valor"])

,participant_id,momento,exercicio,valor,ordem
432,91,follow,ex1,5.591250e+05,3.0
459,95,follow,ex1,5.591250e+05,3.0
138,36,pos,ex1,5.040600e+04,2.0
165,42,pos,ex1,5.040600e+04,2.0
231,55,pos,ex1,5.040600e+04,2.0
237,56,pos,ex1,5.040600e+04,2.0
78,16,pos,ex1,5.040600e+05,3.0
84,17,pos,ex1,5.040600e+05,3.0
333,70,pos,ex1,5.040600e+08,6.0
342,71,pos,ex1,5.040600e+08,6.0


In [118]:
CODIGO = {"certo": 1, "errado": 0, "errado_unidade": 0,
          "vazia": 0 if VAZIA_CONTA_COMO_ERRADA else np.nan,
          "rever": np.nan}

respostas["correct"] = respostas["classificacao"].map(CODIGO)
respostas["revisao"] = ""
respostas["nota"] = ""

REVISAO_POR_VALOR = {
    # (momento, exercício, valor): (1 ou 0, categoria, motivo)
    ("pre", "ex1", 4513875):   (1, "virgula_ou_unidade", "451,3875 mg ou 0,4513875 g sem vírgula"),
    ("pre", "ex1", 45139):     (1, "virgula_ou_unidade", "451,39 mg ou 0,45139 g sem vírgula"),
    ("pre", "ex1", 451388):    (1, "virgula_ou_unidade", "451,388 mg, 0,451388 g ou µg"),
    ("pre", "ex1", 451387):    (1, "virgula_ou_unidade", "451,387 mg, 0,451387 g ou µg"),
    ("pre", "ex1", 0.451):     (1, "unidade_aceite", "gramas; unidade não distinguível no ex1"),
    ("pre", "ex1", 4513):      (1, "ambiguo", "451,3 sem vírgula ou erro de fator 10"),
    ("pre", "ex1", 4510):      (1, "ambiguo", "451,0 sem vírgula ou erro de fator 10"),
    ("pre", "ex2", 475):       (1, "virgula_inequivoca", "47,5 sem vírgula (17 alunos)"),
    ("pre", "ex2", 0.0475):    (1, "unidade_aceite", "litros; o enunciado não pedia unidade"),
    ("pre", "ex2", 47505):     (0, "nao_interpretavel", "sem leitura plausível"),
    ("pos", "ex1", 50406):     (1, "virgula_inequivoca", "504,06 sem vírgula"),
    ("pos", "ex1", 504060):    (1, "virgula_ou_unidade", "504,060 mg, 0,504060 g ou µg"),
    ("pos", "ex1", 504060000): (1, "unidade_aceite", "valor certo em ng"),
    ("pos", "ex2", 6):         (0, "erro_calculo", "fator 10 abaixo; não é vírgula nem unidade"),
    ("follow", "ex1", 559125): (1, "virgula_ou_unidade", "559,125 mg, 0,559125 g ou µg sem vírgula")
}


for (mom, ex, val), (decisao, categoria, motivo) in REVISAO_POR_VALOR.items():
    linhas = ((respostas["momento"] == mom) & (respostas["exercicio"] == ex)
              & np.isclose(respostas["valor"], val, rtol=0, atol=1e-6))
    assert linhas.sum() >= 1, f"Não encontrei o valor {val} em {mom}, {ex}"
    assert (respostas.loc[linhas, "classificacao"] == "rever").all(), f"{val} em {mom}, {ex} não estava para rever"
    respostas.loc[linhas, "correct"] = decisao
    respostas.loc[linhas, "revisao"] = categoria
    respostas.loc[linhas, "nota"] = motivo

In [119]:
## ATENÇAO, verificar para follow up tbm quando dados reais
por_decidir = (respostas["classificacao"] == "rever") & (respostas["revisao"] == "")
assert not por_decidir.any(), f"Faltam decidir {por_decidir.sum()} casos"

In [120]:
respostas["correct_estrito"] = (respostas["classificacao"] == "certo").astype(float)

respostas["correct_ambiguos_inv"] = respostas["correct"]
ambiguos = respostas["revisao"] == "ambiguo"
respostas.loc[ambiguos, "correct_ambiguos_inv"] = 1 - respostas.loc[ambiguos, "correct"]

if not VAZIA_CONTA_COMO_ERRADA:
    respostas.loc[respostas["classificacao"] == "vazia", ["correct_estrito", "correct_ambiguos_inv"]] = np.nan

In [121]:
respostas["momento"] = pd.Categorical(respostas["momento"], categories=["pre", "pos", "follow"], ordered=True)
with pd.option_context('display.max_colwidth', None):
    display(
        respostas.groupby(["momento", "exercicio", "classificacao"], observed=True)["valor"].unique()
        )

momento  exercicio  classificacao 
pre      ex1        certo                                                                                                                                           [451.0, 451.4]
                    errado                                                              [3.75, 0.0, 421.0, 450.0, 375.0, 3.5, 45.0, 200.0, 5.0, 6.0, 0.45, 3750.0, 31154.0, 452.0]
                    rever                                                                                          [451388.0, 0.451, 45139.0, 4513.0, 4513875.0, 4510.0, 451387.0]
         ex2        certo                                                                                                                                 [48.0, 47.5, 47.1, 47.4]
                    errado            [45.0, 0.0, 6.25, 225.0, 0.61, 61.0, 625.0, 62.5, 1.0, 624.0, 12.5, 36.0, 23.0, 0.625, 46.0, 60.0, 150.0, 21.0, 0.0036, 63.0, 0.0045, 114.0]
                    rever                                             

#### 6.1.2. Categorizar estrategias de resposta

In [122]:
METODOS = {
    "Fórmula matemática": "formula",
    "Resolvi por etapas": "etapas",
    "Regra de três simples": "regra3",
    "Desenhei um esquema": "esquema",
    "Tentativa e erro": "tentativa_erro",
}

def codificar_metodo(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip()
    for inicio, codigo in METODOS.items():
        if texto.startswith(inicio):
            return codigo
    return "outro"

respostas["strategy"] = respostas["strategy_raw"].apply(codificar_metodo)

In [123]:
assert not ((respostas["strategy"] == "etapas") & (respostas["exercicio"] != "ex3")).any()
pd.crosstab(respostas["exercicio"], respostas["strategy"])

strategy,esquema,etapas,formula,outro,regra3,tentativa_erro
exercicio,,,,,,
ex1,6,0,191,18,16,15
ex2,5,0,175,28,25,13
ex3,12,71,78,44,17,24


In [124]:
respostas["strategy_outro_texto"] = respostas["strategy_raw"].where(respostas["strategy"] == "outro")
with pd.option_context('display.max_colwidth', None):
    display(
        respostas[respostas["strategy"] == "outro"].groupby(["momento", "exercicio"], observed=True)["strategy_raw"].unique()
    )

momento  exercicio
pre      ex1                                                    [não sei , Através de fórmulas químicas, ou seja, da concentração e da massa.e, Não Sei , Não sei, Não sei ]
         ex2                                                                                         [Não sei, Nao sei, não sei , n sei, Não sei , Naosei, Não Sei, não sei]
         ex3                                                    [Não sei , Não sei, Nao sei, Não fiz, n sei, Nao sei resolver, não sei , não sei, 8x5ˆ3, nao sei, No lo sie]
pos      ex1                                                            [n sei, Não sei , nao sei, 0, Não tive tempo, não tive tempo de responder , 00, Não sei, Não tenho ]
         ex2                                             [não sei, n sei, Não sei , 0, Não deu tempo, Não sei, Não tive tempo, Não tive tempo de responder , 00, Sem tempo ]
         ex3          [não tive tempo, não sei, n sei, não sei , Não sei , Não sei, nao sei, 0, Não deu tempo, Não t

In [125]:
def recodificar_outro(texto):
    t = str(texto).strip().lower()
    if "tempo" in t:
        return "sem_tempo"
    if "sei" in t or "fiz" in t or "tenho" in t or "no lo" in t or t in ["0", "00"]:
        return "nao_resolveu"
    return None

for i in respostas.index[respostas["strategy"] == "outro"]:
    novo = recodificar_outro(respostas.loc[i, "strategy_raw"])
    if novo is not None:
        respostas.loc[i, "strategy"] = novo

In [126]:
RECODIFICAR_OUTRO = {
    "Através de fórmulas químicas, ou seja, da concentração e da massa.e": "formula",
    "8x5ˆ3" : "formula"
}

for texto, codigo in RECODIFICAR_OUTRO.items():
    linhas = (respostas["strategy"] == "outro") & (respostas["strategy_raw"].str.strip() == texto)
    assert linhas.sum() >= 1, f"Não encontrei: {texto}"
    respostas.loc[linhas, "strategy"] = codigo

In [127]:
respostas[respostas["strategy"] == "outro"][["participant_id", "momento", "exercicio", "strategy_raw"]]

,participant_id,momento,exercicio,strategy_raw


In [128]:
pd.crosstab([respostas["momento"], respostas["exercicio"]], respostas["strategy"])

strategy           esquema  etapas  formula  nao_resolveu  regra3  sem_tempo  \
momento exercicio                                                              
pre     ex1              3       0       90             6       6          0   
        ex2              1       0       77            14      13          0   
        ex3              7      37       24            25       5          0   
pos     ex1              3       0       62             9       6          2   
        ex2              4       0       59            10       7          4   
        ex3              2      14       38            13       9          5   
follow  ex1              0       0       40             0       4          0   
        ex2              0       0       39             0       5          0   
        ex3              3      20       17             0       3          0   

strategy           tentativa_erro  
momento exercicio                  
pre     ex1                     8  
        ex2                     8  
        ex3                    15  
pos     ex1                     7  
        ex2                     5  
        ex3                     8  
follow  ex1                     0  
        ex2                     0  
        ex3                     1

In [129]:
# Instrução da investigadora: responder 0 e "não sei" quando não conseguissem resolver em tempo útil
respostas["nao_resolvido"] = (np.isclose(respostas["valor"], 0, rtol=0, atol=1e-6)
                              | respostas["strategy"].isin(["nao_resolveu", "sem_tempo"])).astype(int)

In [130]:
ids_pos_sem_tempo = respostas.loc[(respostas["momento"] == "pos") & (respostas["strategy"] == "sem_tempo"),
                                  "participant_id"].unique()
print(len(ids_pos_sem_tempo), sorted(ids_pos_sem_tempo))

5 [np.int64(7), np.int64(66), np.int64(96), np.int64(99), np.int64(113)]


#### 6.1.3. Marcar respostas iguais no pre e no pos (o estudante nao leu o enunciado)

In [131]:
#A resposta certa do set do pré aparece no pós
def resposta_do_pre(linha):
    if linha["momento"] == "pre" or pd.isna(linha["valor"]):
        return np.nan
    certa_pre = CERTAS["pre"][linha["exercicio"]]
    return 0 if pd.isna(ordem_grandeza(linha["valor"], certa_pre)) else 1

respostas["resposta_do_pre"] = respostas.apply(resposta_do_pre, axis=1)

In [132]:
#O aluno repetiu a sua própria resposta do pré
valores_pre = respostas[respostas["momento"] == "pre"][["participant_id", "exercicio", "valor"]]
valores_pre = valores_pre.rename(columns={"valor": "valor_pre"})

n_antes = len(respostas)
respostas = respostas.merge(valores_pre, on=["participant_id", "exercicio"], how="left")
assert len(respostas) == n_antes, "O merge duplicou linhas"

respostas["igual_ao_proprio_pre"] = (np.isclose(respostas["valor"], respostas["valor_pre"], rtol=0, atol=1e-6)).astype(float)
respostas.loc[respostas["momento"] == "pre", "igual_ao_proprio_pre"] = np.nan
respostas = respostas.drop(columns="valor_pre")

#### 6.1.4. Preparação para deteção de colaboração

In [133]:
TRIVIAIS = [0, 1]
DILUICOES_ERRADAS = {"pre": [40, 200, 120], "pos": [60, 240, 180], "follow": [50, 500, 150]}

def erro_comum(linha):
    v = linha["valor"]
    certa = CERTAS[linha["momento"]][linha["exercicio"]]
    if any(np.isclose(v, t, rtol=0, atol=1e-6) for t in TRIVIAIS):
        return "trivial"
    if linha["exercicio"] == "ex2" and linha["volume_final"] == 1:
        return "volume_final"
    for fator in [1000, 1 / 1000]:
        if abs(v * fator - certa) <= 0.01 * certa:
            return "unidade"
    if linha["exercicio"] == "ex3" and any(np.isclose(v, d, rtol=0, atol=1e-6)
                                           for d in DILUICOES_ERRADAS[linha["momento"]]):
        return "diluicoes"
    return ""

respostas["erro_comum"] = respostas.apply(erro_comum, axis=1)

In [134]:
LIMITE_RARO = 3   # DECISÃO: uma resposta "rara" é dada por no máximo 3 alunos

invulgares = respostas[(respostas["classificacao"] != "certo") & respostas["valor"].notna()
                       & (respostas["erro_comum"] == "")]

partilhas = (invulgares.groupby(["momento", "exercicio", "valor"], observed=True)["participant_id"]
             .agg(n="count", ids=lambda s: sorted(s)).reset_index())

partilhas_raras = partilhas[(partilhas["n"] >= 2) & (partilhas["n"] <= LIMITE_RARO)]
partilhas_raras

,momento,exercicio,valor,n,ids
12,pre,ex1,4.513000e+03,2,"[64, 65]"
22,pre,ex2,3.600000e+01,2,"[49, 146]"
23,pre,ex2,4.500000e+01,2,"[1, 9]"
26,pre,ex2,6.100000e+01,3,"[13, 34, 35]"
32,pre,ex3,3.200000e-01,2,"[41, 69]"
51,pos,ex1,5.040600e+08,3,"[70, 71, 73]"
53,pos,ex2,6.000000e+00,3,"[36, 100, 147]"
74,follow,ex1,5.220000e+02,2,"[34, 39]"


In [135]:
partilhas_raras.to_csv( PASTA / "dados_limpos" / "partilhas_raras.csv", index=False)

#### 6.1.5. Output Tabela Respostas

Nota! Acrescentar follow up nos outputs depois

In [136]:
respostas = respostas.sort_values(["participant_id", "momento", "exercicio"]).reset_index(drop=True)

assert not respostas.duplicated(["participant_id", "momento", "exercicio"]).any(), "Há linhas repetidas"
assert respostas["correct"].notna().all(), "Há respostas sem cotação"

respostas["momento"] = respostas["momento"].cat.remove_unused_categories()
respostas.to_parquet(PASTA / "dados_limpos" / "respostas.parquet")

##### **Tabela de respostas: decisões e pendentes**

#### Tabela de respostas: decisões e pendentes

**Estrutura**
- Uma linha por participante × momento × exercício (pré, pós e follow-up).

**Decisões tomadas**
- Tolerância de ±0,5 sobre o valor exato (aceita arredondamentos, incluindo intermédios; ex.: 47,1 e 47,4 aceites no ex2 do pré). Aplicada da mesma forma nos três momentos: 452 (pré) e 560/561 (follow-up) contam como errados.
- Vírgula decimal perdida na submissão do Forms (nos três formulários; não recuperável). Recuperação decidida por valor (`REVISAO_POR_VALOR`), com categoria e motivo. No follow-up: 559125 (2 casos) aceite como 559,125 mg.
- Unidades:
  - ex1: qualquer unidade de massa aceite (g e mg indistinguíveis depois da perda da vírgula);
  - ex2: qualquer unidade aceite (o enunciado não pedia unidade);
  - ex3: o enunciado pede unidade; uma resposta noutra unidade é errada. No follow-up (fator 1:10), um erro de unidade é indistinguível de contar só 2 diluições (ex.: 500): a cotação não muda, só o rótulo do tipo de erro é ambíguo.
- Ambíguos (vírgula vs. erro de fator 10: 4513 e 4510 no pré) aceites como certos na análise principal.
- Respostas em branco contam como erradas; resposta 0 ou "não sei" = `nao_resolvido`.
- Cotação por campo: uma resposta escrita na caixa de outro exercício conta como errada.
- Métodos codificados pelo início do texto (formula, etapas [só ex3], regra3, esquema, tentativa_erro). "Outro" recodificado por palavras-chave (nao_resolveu, sem_tempo); 2 textos do pré recodificados para formula. Sem "Outro" no follow-up.
- `sem_tempo` é tratado como em falta nas análises de métodos; `nao_resolveu` é uma categoria.
- Colaboração: respostas invulgares (não certas, fora da lista de erros comuns) dadas por 2 a 3 alunos (`LIMITE_RARO = 3`) → `partilhas_raras.csv`. O critério de sinalização está na tabela de participantes.

**Variáveis para análises de sensibilidade**
- `correct_estrito`: só respostas certas tal como registadas, sem recuperações.
- `correct_ambiguos_inv`: decisão sobre os ambíguos invertida.

**Marcações**
- `resposta_do_pre`, `igual_ao_proprio_pre`, `volume_final` (ex2), `erro_comum`, `nao_resolvido`.

**Pendentes**
- Nenhum na limpeza. A contagem das respostas do pré repetidas no pós e dos tipos de erro (para a discussão) passa para o notebook de análise.

In [137]:
respostas["momento"].value_counts()

momento
pre       339
pos       267
follow    132
Name: count, dtype: int64

### 6.2. Tabela Escalas

#### 6.2.1. Calcular Scores

In [138]:
AFETO = [f"affect_{i}" for i in range(1, 11)]
MOTIV = [f"motiv_{i}" for i in range(1, 17)]
ITENS = AFETO + MOTIV

pre_esc = df_pre[["participant_id"] + ITENS].copy()
pre_esc["momento"] = "pre"

pos_esc = df_pos[["participant_id"] + ITENS].copy()
pos_esc["momento"] = "pos"

escalas = pd.concat([pre_esc, pos_esc], ignore_index=True)
assert not escalas.duplicated(["participant_id", "momento"]).any(), "Há linhas repetidas"

In [139]:
escalas


,participant_id,affect_1,affect_2,affect_3,affect_4,affect_5,affect_6,affect_7,affect_8,affect_9,...,motiv_8,motiv_9,motiv_10,motiv_11,motiv_12,motiv_13,motiv_14,motiv_15,motiv_16,momento
0,1,3,1,4,1,2,2,2,1,4,...,3.0,4.0,3.0,3.0,4.0,3.0,4.0,3.0,3.0,pre
1,2,5,1,5,2,5,5,2,1,5,...,2.0,7.0,1.0,6.0,1.0,7.0,7.0,6.0,1.0,pre
2,4,4,2,4,1,3,3,1,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pre
3,5,5,3,4,2,5,4,2,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pre
4,7,3,2,4,1,4,4,1,1,3,...,5.0,6.0,4.0,4.0,2.0,4.0,6.0,6.0,3.0,pre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,145,4,4,3,4,4,3,4,2,3,...,4.0,4.0,5.0,5.0,4.0,4.0,4.0,5.0,6.0,pos
198,146,4,2,4,1,4,4,1,1,4,...,2.0,5.0,5.0,1.0,1.0,5.0,5.0,2.0,2.0,pos
199,147,4,3,4,2,3,4,2,2,5,...,2.0,5.0,5.0,3.0,2.0,5.0,6.0,5.0,3.0,pos
200,148,5,3,4,1,3,3,1,1,5,...,1.0,7.0,7.0,4.0,1.0,7.0,7.0,7.0,1.0,pos


In [140]:
print("Tipos:", escalas[ITENS].dtypes.unique())
print("PANAS: mínimo", escalas[AFETO].min().min(), "| máximo", escalas[AFETO].max().max())
print("SIMS:  mínimo", escalas[MOTIV].min().min(), "| máximo", escalas[MOTIV].max().max())
print("Itens em falta:", escalas[ITENS].isna().sum().sum())

Tipos: [dtype('int64') dtype('float64')]
PANAS: mínimo 1 | máximo 5
SIMS:  mínimo 1.0 | máximo 7.0
Itens em falta: 39


In [141]:
escalas.groupby("momento")[MOTIV].apply(lambda x: x.isna().sum().sum())

momento
pos     0
pre    39
dtype: int64

In [142]:
pre_motiv = escalas[escalas["momento"] == "pre"][MOTIV]
print(pre_motiv.isna().sum())          # em falta por item
print(pre_motiv.isna().sum(axis=1).value_counts().sort_index())   # em falta por aluno

motiv_1     1
motiv_2     3
motiv_3     3
motiv_4     2
motiv_5     2
motiv_6     3
motiv_7     3
motiv_8     2
motiv_9     2
motiv_10    3
motiv_11    2
motiv_12    2
motiv_13    2
motiv_14    2
motiv_15    3
motiv_16    4
dtype: int64
0     103
1       8
15      1
16      1
Name: count, dtype: int64


In [143]:
AP_ITENS = ["affect_1", "affect_3", "affect_5", "affect_6", "affect_9"]   # afeto positivo
AN_ITENS = ["affect_2", "affect_4", "affect_7", "affect_8", "affect_10"]  # afeto negativo
assert sorted(AP_ITENS + AN_ITENS) == sorted(AFETO), "Os itens da PANAS não estão todos atribuídos"

SIMS_SUBESCALAS = {
    "motivacao_intrinseca":   ["motiv_1", "motiv_5", "motiv_9",  "motiv_13"],
    "regulacao_identificada": ["motiv_2", "motiv_6", "motiv_10", "motiv_14"],
    "regulacao_externa":      ["motiv_3", "motiv_7", "motiv_11", "motiv_15"],
    "desmotivacao":           ["motiv_4", "motiv_8", "motiv_12", "motiv_16"],
}

escalas["AP"] = escalas[AP_ITENS].sum(axis=1, min_count=5)
escalas["AN"] = escalas[AN_ITENS].sum(axis=1, min_count=5)

for nome, itens in SIMS_SUBESCALAS.items():
    completo = escalas[itens].notna().all(axis=1)
    escalas[nome] = escalas[itens].mean(axis=1).where(completo)

In [144]:
print(escalas[["AP", "AN"] + list(SIMS_SUBESCALAS)].describe().loc[["count", "min", "max"]])

          AP     AN  motivacao_intrinseca  regulacao_identificada  \
count  202.0  202.0                 200.0                   197.0   
min      5.0    5.0                   1.0                     1.0   
max     25.0   25.0                   7.0                     7.0   

       regulacao_externa  desmotivacao  
count              197.0         198.0  
min                  1.0           1.0  
max                  7.0           7.0  


#### 6.2.2. Output Tabela Escalas

In [145]:
escalas["momento"] = pd.Categorical(escalas["momento"], categories=["pre", "pos"], ordered=True)
escalas = escalas.sort_values(["participant_id", "momento"]).reset_index(drop=True)
escalas.to_parquet(PASTA / "dados_limpos" / "escalas.parquet")

##### **Tabela de escalas: decisões e pendentes**

**Estrutura**
- Uma linha por participante × momento (pré: 113; pós: 89; total: 202).
- Mantém os itens individuais (affect_1–10, motiv_1–16) para a análise de fiabilidade.

**Decisões tomadas**
- Itens numerados pela ordem do formulário.
- PANAS: afeto positivo (AP) = soma de affect_1, 3, 5, 6, 9; afeto negativo (AN) =
  soma de affect_2, 4, 7, 8, 10. Amplitude de cada score: 5–25.
- SIMS: escala de 1 a 7 (confirmada nos dados). Subescalas calculadas como a média de 4 itens:
  - motivação intrínseca: motiv_1, 5, 9, 13;
  - regulação identificada: motiv_2, 6, 10, 14;
  - regulação externa: motiv_3, 7, 11, 15;
  - desmotivação: motiv_4, 8, 12, 16.
- Itens em falta: o score só é calculado se todos os itens da (sub)escala estiverem
  respondidos; sem imputação (alternativa considerada: média com ≥3 de 4 itens; rejeitada
  por simplicidade; perda máxima de 5 scores por subescala).

**Dados em falta**
- PANAS: 0 itens em falta; 202 scores calculados.
- SIMS: 39 itens em falta, todos no pré (nenhum no pós; os itens no pré
  provavelmente não eram obrigatórios — confirmar no Forms):
  - 8 alunos com 1 item em falta (dispersos por itens diferentes);
  - 2 alunos sem resposta à SIMS (15 e 16 itens em falta).
- Regra: o score da subescala só é calculado com os 4 itens respondidos; sem imputação.
- Scores de subescala em falta (pré): motivação intrínseca 2, regulação identificada 5,
  regulação externa 5, desmotivação 4.

  
**Pendentes**

- Nenhum

### 6.3. Tabela Participantes

#### 6.3.1. Calcular SUS

In [146]:
SUS = [f"sus{i}" for i in range(1, 11)]

for col in SUS:
     if not pd.api.types.is_numeric_dtype(df_pos[col]):
        df_pos[col] = pd.to_numeric(df_pos[col].astype(str).str.extract(r"^(\d+)", expand=False), errors="coerce")

print("SUS: mínimo", df_pos[SUS].min().min(), "| máximo", df_pos[SUS].max().max(),
      "| em falta", df_pos[SUS].isna().sum().sum())

sus_impares = ["sus1", "sus3", "sus5", "sus7", "sus9"]    # frases positivas
sus_pares = ["sus2", "sus4", "sus6", "sus8", "sus10"]     # frases negativas
contribuicao = (df_pos[sus_impares] - 1).sum(axis=1, min_count=5) + (5 - df_pos[sus_pares]).sum(axis=1, min_count=5)
df_pos["SUS"] = contribuicao * 2.5

SUS: mínimo 1 | máximo 5 | em falta 0


#### 6.3.2. Preparar variaveis descritoras da experiencia de jogo

In [147]:
df_pos["lp_total"] = df_pos[["lp_level1", "lp_level2", "lp_level3"]].sum(axis=1, min_count=3)
df_pos["exercicios_disponiveis"] = df_pos["final_level"] * 3
df_pos["lp_pct"] = df_pos["lp_total"] / df_pos["exercicios_disponiveis"] * 100
assert (df_pos["lp_pct"].dropna() <= 100).all(), "lp_pct acima de 100%"

LIMITE_MIN = 5   # DECISÃO: menos de 5 minutos de jogo = provável recarga
df_pos["provavel_recarga"] = (df_pos["game_time_min"] < LIMITE_MIN).astype(float)
df_pos.loc[df_pos["game_time_min"].isna(), "provavel_recarga"] = np.nan

In [148]:
df_pos["pos_sem_tempo"] = df_pos["participant_id"].isin(ids_pos_sem_tempo).astype(int)

#### 6.3.3. Juntar a base

In [149]:
ids = sorted(set(df_pre["participant_id"]) | set(df_pos["participant_id"]) | set(df_follow_up["participant_id"]))
participantes = pd.DataFrame({"participant_id": ids})

participantes["tem_pre"] = participantes["participant_id"].isin(df_pre["participant_id"]).astype(int)
participantes["tem_pos"] = participantes["participant_id"].isin(df_pos["participant_id"]).astype(int)
participantes["tem_follow"] = participantes["participant_id"].isin(df_follow_up["participant_id"]).astype(int)
participantes["par_pre_pos"] = ((participantes["tem_pre"] == 1) & (participantes["tem_pos"] == 1)).astype(int)

ids_nao_emp = (set(df_pre.loc[df_pre["pre_pos_nao_emparelhavel"] == 1, "participant_id"])
               | set(df_pos.loc[df_pos["pre_pos_nao_emparelhavel"] == 1, "participant_id"]))
participantes["pre_pos_nao_emparelhavel"] = participantes["participant_id"].isin(ids_nao_emp).astype(int)
participantes["follow_nao_atribuivel"] = participantes["participant_id"].isin([1131, 1132]).astype(int)

# Amostras da família 3
valido_follow = (participantes["tem_follow"] == 1) & (participantes["follow_nao_atribuivel"] == 0)
pre_valido_ = (participantes["tem_pre"] == 1) & (participantes["participant_id"] != 591) & (participantes["participant_id"] != 592)
participantes["par_pre_follow"] = (valido_follow & pre_valido_).astype(int)                     # F1
participantes["tres_momentos"] = (valido_follow & (participantes["par_pre_pos"] == 1)).astype(int)  # F2

assert participantes["par_pre_pos"].sum() == fluxogram["pares_pre_pos"]
assert participantes["par_pre_follow"].sum() == fluxogram["follow_com_pre"]
assert participantes["tres_momentos"].sum() == fluxogram["follow_3_momentos"]

In [150]:
# colocar os turnos
turno_pre = df_pre.set_index("participant_id")["turno"]
turno_pos = df_pos.set_index("participant_id")["turno"]

participantes["turno"] = participantes["participant_id"].map(turno_pre)
participantes["turno"] = participantes["turno"].fillna(participantes["participant_id"].map(turno_pos))
assert participantes.loc[(participantes["tem_pre"] == 1) | (participantes["tem_pos"] == 1), "turno"].notna().all()

In [151]:
COLS_PRE = ["participant_id", "gender", "prior_education", "digital_confort", "ia_confort", "fill_in_time"]
pre_part = df_pre[COLS_PRE].rename(columns={"fill_in_time": "pre_fill_time"})

n = len(participantes)
participantes = participantes.merge(pre_part, on="participant_id", how="left")
assert len(participantes) == n, "O merge duplicou linhas"

In [152]:
participantes

,participant_id,tem_pre,tem_pos,tem_follow,par_pre_pos,pre_pos_nao_emparelhavel,follow_nao_atribuivel,par_pre_follow,tres_momentos,turno,gender,prior_education,digital_confort,ia_confort,pre_fill_time
0,1,1,1,0,1,0,0,0,0,1.0,0.0,0.0,2.0,2.0,17.33
1,2,1,1,0,1,0,0,0,0,1.0,0.0,0.0,5.0,5.0,13.13
2,4,1,0,0,0,0,0,0,0,1.0,0.0,0.0,4.0,3.0,18.53
3,5,1,1,1,1,0,0,1,1,1.0,1.0,0.0,3.0,4.0,18.13
4,7,1,1,1,1,0,0,1,1,2.0,0.0,0.0,4.0,3.0,18.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,149,1,1,0,1,0,0,0,0,3.0,1.0,0.0,5.0,4.0,11.83
114,591,1,0,0,0,1,0,0,0,1.0,0.0,0.0,2.0,3.0,9.05
115,592,1,0,0,0,1,0,0,0,1.0,0.0,0.0,4.0,3.0,7.03
116,1131,0,0,1,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [153]:
#Juntar tudo: VAI ENTRAR AQUI FOLLOW UP DEPOIS
COLS_POS = (["participant_id", "fill_in_time", "pos_fora_sessao", "pos_sem_tempo", "SUS"] + SUS
            + ["game_results", "game_finished", "final_level", "arrived_level3", "xp", "estabilidade",
               "game_time_min", "lp_level1", "lp_level2", "lp_level3", "lp_total", "lp_pct",
               "provavel_recarga", "nota_jogo",
               "labpartner_function", "lp_deu_codigo", "lp_fez_calculos", "lp_confuso",
               "lp_funcionou_bem", "lp_problema", "lp_outro_texto", "lp_outro_cat",
               "lp_problema_tecnico", "lp_nao_usou", "lp_outro",
               "general_comment", "positive_comment", "negative_comment"])
pos_part = df_pos[COLS_POS].rename(columns={"fill_in_time": "pos_fill_time"})

n = len(participantes)
participantes = participantes.merge(pos_part, on="participant_id", how="left")
assert len(participantes) == n, "O merge duplicou linhas"

In [154]:
COLS_FOLLOW = (["participant_id", "fill_in_time", "dias_desde_sessao",
                "more_practice", "practice_source", "practiced", "perceived_usefulness",
                "tech_used", "tech_phone", "tech_tablet", "tech_computer", "tech_dont_remember", "tech_n",
                "ai_use_raw", "ai_access", "ai_use", "reason_not_ai_use"]
               + list(MOTIVOS.values()) + ["motivo_outro", "motivo_outro_texto", "comments"])
follow_part = df_follow_up[COLS_FOLLOW].rename(columns={"fill_in_time": "follow_fill_time",
                                                        "comments": "follow_comment"})
n = len(participantes)
participantes = participantes.merge(follow_part, on="participant_id", how="left")
assert len(participantes) == n, "O merge duplicou linhas"

#### 6.3.4. Marcar colaboração entre participantes para analise de sensibilidade

In [155]:
LIMITE_MIN_COLAB = 5   # DECISÃO 19: minutos entre submissões

# hora de submissão e turno de cada participante em cada momento
tempos = pd.concat([
    df_pre[["participant_id", "end_time", "turno"]].assign(momento="pre"),
    df_pos[["participant_id", "end_time", "turno"]].assign(momento="pos"),
    df_follow_up[["participant_id", "end_time"]].assign(turno=0, momento="follow"),
]).set_index(["participant_id", "momento"])

In [156]:
linhas = []
for _, partilha in partilhas_raras.iterrows():
    momento = str(partilha["momento"])
    for a, b in combinations(partilha["ids"], 2):
        ta, tb = tempos.loc[(a, momento)], tempos.loc[(b, momento)]
        minutos = abs((ta["end_time"] - tb["end_time"]).total_seconds()) / 60
        mesmo_turno = ta["turno"] == tb["turno"]
        linhas.append({"momento": momento, "exercicio": partilha["exercicio"], "valor": partilha["valor"],
                       "aluno_a": a, "aluno_b": b, "mesmo_turno": mesmo_turno,
                       "minutos": round(minutos, 1),
                       "sinalizado": mesmo_turno and minutos <= LIMITE_MIN_COLAB})

pares_colab = pd.DataFrame(linhas)
pares_colab

,momento,exercicio,valor,aluno_a,aluno_b,mesmo_turno,minutos,sinalizado
0,pre,ex1,4.513000e+03,64,65,True,1.5,True
1,pre,ex2,3.600000e+01,49,146,False,84.7,False
2,pre,ex2,4.500000e+01,1,9,True,1.1,True
3,pre,ex2,6.100000e+01,13,34,False,82.0,False
4,pre,ex2,6.100000e+01,13,35,False,81.9,False
5,pre,ex2,6.100000e+01,34,35,True,0.1,True
6,pre,ex3,3.200000e-01,41,69,True,7.3,False
7,pos,ex1,5.040600e+08,70,71,True,0.7,True
8,pos,ex1,5.040600e+08,70,73,True,0.6,True
9,pos,ex1,5.040600e+08,71,73,True,0.1,True


In [157]:
respostas.loc[np.isclose(respostas["valor"], 1, rtol=0, atol=1e-6)
              & (respostas["momento"] == "pos"), ["participant_id", "exercicio", "valor", "strategy"]]

,participant_id,exercicio,valor,strategy
27,7,ex1,1.0,formula
42,9,ex1,1.0,formula
56,12,ex3,1.0,regra3
105,27,ex1,1.0,regra3
106,27,ex2,1.0,formula
107,27,ex3,1.0,tentativa_erro


In [158]:
sinal = pares_colab[pares_colab["sinalizado"]]
ids_colab = set(sinal.loc[sinal["momento"].isin(["pre", "pos"]), ["aluno_a", "aluno_b"]].values.ravel())
ids_colab_follow = set(sinal.loc[sinal["momento"] == "follow", ["aluno_a", "aluno_b"]].values.ravel())

participantes["suspeita_colaboracao"] = participantes["participant_id"].isin(ids_colab).astype(int)
participantes["suspeita_colaboracao_follow"] = participantes["participant_id"].isin(ids_colab_follow).astype(int)
print("Pré/pós:", sorted(ids_colab), "| Follow-up:", sorted(ids_colab_follow))
print(participantes.groupby("turno")["suspeita_colaboracao"].sum())
pares_colab.to_csv(PASTA / "dados_limpos" / "pares_colaboracao.csv", index=False)

Pré/pós: [np.int64(1), np.int64(9), np.int64(34), np.int64(35), np.int64(64), np.int64(65), np.int64(70), np.int64(71), np.int64(73)] | Follow-up: [np.int64(34), np.int64(39)]
turno
1.0    5
2.0    4
3.0    0
Name: suspeita_colaboracao, dtype: int64


#### 6.3.5. Output tabela participantes

In [159]:
assert participantes["participant_id"].is_unique
print(participantes[["tem_pre", "tem_pos", "par_pre_pos", "pre_pos_nao_emparelhavel",
                     "pos_fora_sessao", "pos_sem_tempo"]].sum())
print(participantes[["SUS", "lp_pct"]].describe().loc[["count", "min", "max"]])

participantes.to_parquet(PASTA / "dados_limpos" / "participantes.parquet")

tem_pre                     113.0
tem_pos                      89.0
par_pre_pos                  87.0
pre_pos_nao_emparelhavel      4.0
pos_fora_sessao               2.0
pos_sem_tempo                 5.0
dtype: float64
         SUS  lp_pct
count   89.0    61.0
min     32.5     0.0
max    100.0   100.0


##### **Tabela de participantes: decisões e pendentes**

#### Tabela de participantes: decisões e pendentes

**Estrutura**
- Uma linha por participante: união dos códigos do pré, do pós e do follow-up.
- Flags de presença: `tem_pre`, `tem_pos`, `tem_follow`.
- Amostras de análise:
  - `par_pre_pos`: análise primária (87);
  - `par_pre_follow`: família 3, análise principal (F1; 41);
  - `tres_momentos`: família 3, exploratória (F2; 32).
- Códigos não atribuíveis:
  - `pre_pos_nao_emparelhavel`: 591 e 592 (pré) e 58 e 59 (pós). O pós do 58 e do 59 seria emparelhável com o follow-up (F9), mas nenhum dos dois respondeu;
  - `follow_nao_atribuivel`: 1131 e 1132 (duas submissões diferentes com o código 113, provavelmente 113 e 133).
- Turno: atribuído a partir do pré; para quem não tem pré, a partir do pós. Fica vazio para quem só tem follow-up.

**Variáveis do pré**
- `gender`, `prior_education`, `digital_confort`, `ia_confort`, `pre_fill_time`.

**Variáveis do pós**
- `pos_fill_time`, `pos_fora_sessao` (2: códigos 5 e 89), `pos_sem_tempo` (5).
- SUS: itens sus1–sus10 e score [Σ(ímpares − 1) + Σ(5 − pares)] × 2,5, calculado só com os 10 itens.
- Jogo: `game_finished`, `final_level`, `arrived_level3`, `xp`, `estabilidade`, `game_time_min`, `lp_level1–3`, `lp_total`, `lp_pct`, `provavel_recarga`, `nota_jogo`.
- LabPartner durante o jogo: `lp_deu_codigo`, `lp_fez_calculos`, `lp_confuso`, `lp_funcionou_bem`, `lp_problema`; "Outro" codificado em `lp_problema_tecnico`, `lp_nao_usou`, `lp_outro`.
- Comentários abertos.

**Variáveis do follow-up**
- `follow_fill_time`, `dias_desde_sessao` (19–22 dias).
- Estudo: `more_practice`, `practice_source`, `practiced`.
- `perceived_usefulness` (1–5; a pergunta refere-se à **sessão** DILU-IA).
- Dispositivo (escolha múltipla; "computador" = portátil pessoal): `tech_phone`, `tech_tablet`, `tech_computer`, `tech_dont_remember`, `tech_n`.
- Acesso ao LabPartner (autorrelato retrospetivo): `ai_use_raw`, `ai_access`, `ai_use`.
- Motivos para não usar (escolha múltipla): `motivo_sozinho`, `motivo_sem_acesso`, `motivo_sem_tempo`, `motivo_ajuda_humana`, `motivo_nao_percebi`, `motivo_outro` (2 críticas ao estilo de resposta do LabPartner; texto em `motivo_outro_texto`). Só 13 dos 31 que podiam responder indicaram um motivo: reportar com o denominador.
- `follow_comment`.

**Colaboração**
- `suspeita_colaboracao` (pré/pós): par no mesmo momento com ≥ 1 resposta rara partilhada, mesmo turno e submissões com ≤ 5 min de diferença (decisão 19). Sinalizados: 9 (turno 1: 5; turno 2: 4; turno 3: 0).
- `suspeita_colaboracao_follow` (F13): o mesmo critério sem a condição do turno. Sinalizados: 34 e 39.

**Limitações a reportar**
- XP deduzido por erros: não permite contar exercícios.
- Recargas apagam o tempo, o XP e as contagens do LabPartner: `lp_pct` é um mínimo.
- A métrica do LabPartner conta cliques, não uso real.
- O acesso e o dispositivo no follow-up são autorrelato com 3 semanas de distância.

**Pendentes**
- Nenhum na limpeza. Passam para o notebook de análise (família 4 e discussão): coerência entre `lp_nao_usou`, as contagens do jogo e `ai_access`; cruzamento de `lp_deu_codigo` com `arrived_level3` e `game_finished`; contagem dos indicadores de fidelidade.

### 6.4. Tabela Mestra

In [160]:
# Conhecimento a partir da tabela de respostas
conhecimento = (respostas.groupby(["participant_id", "momento"], observed=True)
                .agg(conhecimento=("correct", "sum"),
                     conhecimento_estrito=("correct_estrito", "sum"),
                     conhecimento_amb_inv=("correct_ambiguos_inv", "sum"),
                     n_exercicios=("correct", "count"),
                     n_nao_resolvidos=("nao_resolvido", "sum"))
                .reset_index())

assert (conhecimento["n_exercicios"] == 3).all(), "Há participantes sem os 3 exercícios"
conhecimento = conhecimento.drop(columns="n_exercicios")

In [161]:
# Juntar as escalas
ESCALAS_SCORES = ["AP", "AN", "motivacao_intrinseca", "regulacao_identificada",
                  "regulacao_externa", "desmotivacao"]

mestra = conhecimento.merge(escalas[["participant_id", "momento"] + ESCALAS_SCORES],
                            on=["participant_id", "momento"], how="left", indicator=True)
mestra["momento"] = mestra["momento"].astype(str)
assert (mestra.loc[mestra["momento"] != "follow", "_merge"] == "both").all(), "Pré/pós sem escalas"
mestra = mestra.drop(columns="_merge")

In [162]:
#tempo de preenchimento e turno
tempos = pd.concat([
    df_pre[["participant_id", "fill_in_time"]].assign(momento="pre"),
    df_pos[["participant_id", "fill_in_time"]].assign(momento="pos"),
    df_follow_up[["participant_id", "fill_in_time"]].assign(momento="follow"),
])
tempos["momento"] = tempos["momento"].astype(str)
mestra["momento"] = mestra["momento"].astype(str)

n = len(mestra)
mestra = mestra.merge(tempos, on=["participant_id", "momento"], how="left")
mestra = mestra.merge(participantes[["participant_id", "turno"]], on="participant_id", how="left")
assert len(mestra) == n, "O merge duplicou linhas"

#### 6.4.1. Output tabela mestra

In [163]:
mestra["momento"] = pd.Categorical(mestra["momento"], categories=["pre", "pos", "follow"], ordered=True)
mestra = mestra.sort_values(["participant_id", "momento"]).reset_index(drop=True)

assert not mestra.duplicated(["participant_id", "momento"]).any()
print(mestra["momento"].value_counts())
print(mestra[["conhecimento"] + ESCALAS_SCORES].describe().loc[["count", "min", "max"]])

mestra.to_parquet(PASTA / "dados_limpos" / "mestra.parquet")

momento
pre       113
pos        89
follow     44
Name: count, dtype: int64
       conhecimento     AP     AN  motivacao_intrinseca  \
count         246.0  202.0  202.0                 200.0   
min             0.0    5.0    5.0                   1.0   
max             3.0   25.0   25.0                   7.0   

       regulacao_identificada  regulacao_externa  desmotivacao  
count                   197.0              197.0         198.0  
min                       1.0                1.0           1.0  
max                       7.0                7.0           7.0  


##### **Tabela mestra: decisões e pendentes**

**Estrutura**
- Uma linha por participante × momento (pré, pós e follow-up).

**Variáveis**
- `conhecimento`, `conhecimento_estrito`, `conhecimento_amb_inv` (0–3), `n_nao_resolvidos` (0–3): nos três momentos.
- AP, AN e as 4 subescalas da SIMS: só no pré e no pós (vazias no follow-up).
- `fill_in_time`, `turno`.

**Construção**
- O conhecimento vem da tabela de respostas e as escalas da tabela de escalas; nenhum score é recalculado aqui.
- Verificado: 3 exercícios por participante × momento; todas as linhas do pré e do pós têm escalas.

**Pendentes**
- Nenhum.

### 6.5. Tabela Larga

In [164]:
#pôr momentos lado a lado
VARS_LARGA = ["conhecimento", "conhecimento_estrito", "conhecimento_amb_inv","n_nao_resolvidos" ] + ESCALAS_SCORES

mestra_txt = mestra.copy()
mestra_txt["momento"] = mestra_txt["momento"].astype(str)

larga = mestra_txt.pivot(index="participant_id", columns="momento", values=VARS_LARGA)
larga.columns = [f"{var}_{mom}" for var, mom in larga.columns]
larga = larga.reset_index()
larga = larga.dropna(axis=1, how="all")

In [165]:
# juntar a tabela de participantes
larga = participantes.merge(larga, on="participant_id", how="left")
assert len(larga) == len(participantes), "O merge alterou o número de linhas"

In [166]:
#variaveis de diferenças pre-pos
for var in VARS_LARGA:
    larga[f"delta_{var}"] = larga[f"{var}_pos"] - larga[f"{var}_pre"]

VARS_CONH = ["conhecimento", "conhecimento_estrito", "conhecimento_amb_inv", "n_nao_resolvidos"]
for var in VARS_CONH:
    larga[f"delta_pre_follow_{var}"] = larga[f"{var}_follow"] - larga[f"{var}_pre"]
    larga[f"delta_pos_follow_{var}"] = larga[f"{var}_follow"] - larga[f"{var}_pos"]

In [167]:
for momento in ["pre", "pos", "follow"]:
    col = f"n_nao_resolvidos_{momento}"
    larga[f"sem_resolucao_{momento}"] = (larga[col] == 3).astype(float).where(larga[col].notna())

In [168]:
pares_df = larga[larga["par_pre_pos"] == 1]
assert pares_df[["conhecimento_pre", "conhecimento_pos"]].notna().all().all(), \
    "Há pares sem conhecimento num dos momentos"
assert (larga.loc[larga["pre_pos_nao_emparelhavel"] == 1, "par_pre_pos"] == 0).all(), \
    "Há não emparelháveis marcados como pares"
assert larga.loc[larga["par_pre_follow"] == 1, ["conhecimento_pre", "conhecimento_follow"]].notna().all().all()

print("Participantes:", len(larga), "| Pares:", len(pares_df))

Participantes: 118 | Pares: 87


#### 6.5.1. Output tabela larga

In [169]:
larga.to_parquet(PASTA / "dados_limpos" / "larga.parquet")

In [170]:
larga

,participant_id,tem_pre,tem_pos,tem_follow,par_pre_pos,pre_pos_nao_emparelhavel,follow_nao_atribuivel,par_pre_follow,tres_momentos,turno,...,delta_pos_follow_conhecimento,delta_pre_follow_conhecimento_estrito,delta_pos_follow_conhecimento_estrito,delta_pre_follow_conhecimento_amb_inv,delta_pos_follow_conhecimento_amb_inv,delta_pre_follow_n_nao_resolvidos,delta_pos_follow_n_nao_resolvidos,sem_resolucao_pre,sem_resolucao_pos,sem_resolucao_follow
0,1,1,1,0,1,0,0,0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN
1,2,1,1,0,1,0,0,0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
2,4,1,0,0,0,0,0,0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
3,5,1,1,1,1,0,0,1,1,1.0,...,0.0,2.0,0.0,2.0,0.0,-2.0,0.0,0.0,0.0,0.0
4,7,1,1,1,1,0,0,1,1,2.0,...,2.0,2.0,2.0,1.0,2.0,-1.0,-1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,149,1,1,0,1,0,0,0,0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
114,591,1,0,0,0,1,0,0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
115,592,1,0,0,0,1,0,0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
116,1131,0,0,1,0,0,1,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


##### **Tabela larga: decisões e pendentes**

**Estrutura**
- Uma linha por participante (todos os da tabela de participantes).
- Colunas por momento: `_pre`, `_pos` e `_follow` (no follow-up, só as variáveis de conhecimento; as colunas vazias foram removidas).
- Diferenças:
  - `delta_*` = pós − pré (nomes mantidos dos dados congelados a 11/10);
  - `delta_pre_follow_*` = follow-up − pré;
  - `delta_pos_follow_*` = follow-up − pós.
- `sem_resolucao_pre`, `sem_resolucao_pos`, `sem_resolucao_follow`.

**Verificado**
- Todos os pares pré–pós têm conhecimento no pré e no pós; todos os pares pré–follow-up têm conhecimento no pré e no follow-up.
- Nenhum não emparelhável marcado como par.
- N.º de pares igual ao do fluxograma (87, 41, 32).

**Utilização prevista (filtros)**
- Primária: `par_pre_pos == 1`.
- Sensibilidades pré–pós:
  - `conhecimento_estrito_*`, `conhecimento_amb_inv_*`;
  - `pos_sem_tempo == 0` e `pos_fora_sessao == 0`;
  - `game_finished == 1`;
  - `suspeita_colaboracao == 0`.
- Família 3, principal (F1): `par_pre_follow == 1`.
- Família 3, exploratória (F2): `tres_momentos == 1`; sensibilidade pós vs follow-up com `sem_resolucao_pos == 0` (F3).
- Sensibilidade da colaboração no follow-up: `suspeita_colaboracao_follow == 0`.

**Pendentes**
- Nenhum.

## 7. Dicionário de variáveis

In [171]:
### 7.1. Descrição das variáveis
DESCRICOES = {
    # --- identificação e estrutura ---
    "participant_id": ("Código de participante (591/592 = respostas do pré dos inscritos 58 e 59, "
                       "não atribuíveis; 133 = código reconciliado de '113' no turno 3)", "inteiro"),
    "momento": ("Momento de medição", "pre, pos (follow a acrescentar)"),
    "turno": ("Turno da sessão de 21/09, atribuído pela hora de início do pré "
              "(ou do pós, se não houver pré)", "1, 2, 3"),
 
    # --- flags de participante ---
    "tem_pre": ("Tem questionário pré", "0/1"),
    "tem_pos": ("Tem questionário pós", "0/1"),
    "par_pre_pos": ("Tem pré e pós emparelháveis (amostra da análise primária)", "0/1"),
    "pre_pos_nao_emparelhavel": ("Código partilhado em simultâneo no pré (58/59); pré e pós não atribuíveis", "0/1"),
    "pos_fora_sessao": ("Pós terminado fora da sessão", "0/1"),
    "pos_sem_tempo": ("Indicou no pós que não teve tempo de responder aos exercícios", "0/1"),
    "suspeita_colaboracao" : ("um par de alunos é sinalizado se, no mesmo momento, partilha ≥ 1 resposta invulgar dada por 2 ou 3 alunos, está no mesmo turno e submeteu com ≤ 5 minutos de diferença. É sinalizado o aluno que pertence a pelo menos um par sinalizado. A análise principal inclui todos; a sensibilidade S2 exclui os sinalizados.", "0/1"),
    "erro_comum" : ("marca erros comuns", "trivial, volume_final, unidade, diluicoes ou vazio"),
    "nao_resolvido": ("Resposta 0 ou estratégia de não resolução (por instrução da investigadora: não conseguiu resolver em tempo útil); cotada como errada", "0/1"),
    "n_nao_resolvidos": ("N.º de exercícios não resolvidos", "0–3"),
    "sem_resolucao_pre": ("Nenhum exercício resolvido no pré", "0/1"),
    "sem_resolucao_pos": ("Nenhum exercício resolvido no pós", "0/1"),
    "sem_resolucao_follow": ("Nenhum exercício resolvido no follow-up", "0/1"),
    "tem_follow": ("Tem questionário de follow-up", "0/1"),
    "follow_nao_atribuivel": ("Submissões de follow-up com o código 113 de duas pessoas diferentes (provavelmente 113 e 133); não atribuíveis", "0/1"),
    "par_pre_follow": ("Tem pré e follow-up atribuíveis (amostra principal da família 3, F1)", "0/1"),
    "tres_momentos": ("Tem pré, pós e follow-up (amostra exploratória, F2)", "0/1"),
    "suspeita_colaboracao_follow": ("Critério F13 aplicado ao follow-up (resposta rara partilhada e submissão com ≤ 5 min), sem condição de turno", "0/1"),
 
    # --- demografia (pré) ---
    "gender": ("Género (opções do formulário: Masculino, Feminino, Outro, Prefiro não responder; "
               "todos responderam Feminino ou Masculino)", "0 = feminino, 1 = masculino"),
    "prior_education": ("Já tem um grau superior (opções do formulário: Ensino Secundário, "
                        "Licenciatura, Mestrado, Doutoramento)", "0 = ensino secundário, 1 = licenciatura ou superior"),
    "digital_confort": ("Conforto com videojogos e plataformas digitais", "escala 1–5"),
    "ia_confort": ("Conforto com ferramentas de inteligência artificial", "escala 1–5"),
 
    # --- tempos ---
    "fill_in_time": ("Tempo de preenchimento do questionário nesse momento", "minutos"),
    "pre_fill_time": ("Tempo de preenchimento do pré", "minutos"),
    "pos_fill_time": ("Tempo de preenchimento do pós", "minutos"),
    "follow_fill_time": ("Tempo de preenchimento do follow-up", "minutos"),
    "dias_desde_sessao": ("Dias entre a sessão (21/09) e a resposta ao follow-up", "dias"),
 
    # --- tabela de respostas ---
    "exercicio": ("Exercício de conhecimento", "ex1 (massa), ex2 (volume a adicionar), ex3 (diluição seriada)"),
    "answer_raw": ("Resposta numérica tal como registada pelo Forms (vírgula decimal perdida)", "número"),
    "valor": ("Resposta numérica usada na cotação", "número"),
    "estado": ("Resposta presente ou vazia", "ok, vazia"),
    "ordem": ("k tal que valor = resposta certa × 10^k (±0,5); vazio se não houver relação", "inteiro ou vazio"),
    "classificacao": ("Classificação automática da resposta",
                      "certo, errado, errado_unidade, vazia, rever"),
    "revisao": ("Categoria da decisão manual (casos 'rever')",
                "virgula_inequivoca, virgula_ou_unidade, ambiguo, unidade_aceite, "
                "nao_interpretavel, erro_calculo, unidade_errada"),
    "nota": ("Motivo da decisão manual", "texto"),
    "correct": ("Resposta certa: cotação principal", "0/1"),
    "correct_estrito": ("Resposta certa sem recuperações (só respostas certas tal como registadas)", "0/1"),
    "correct_ambiguos_inv": ("Resposta certa com a decisão sobre os ambíguos invertida", "0/1"),
    "volume_final": ("Ex2: respondeu o volume final em vez do volume a adicionar (descritivo)", "0/1; vazio fora do ex2"),
    "strategy_raw": ("Estratégia de resolução, texto original", "texto"),
    "strategy": ("Estratégia de resolução codificada",
                 "formula, etapas (só ex3), regra3, esquema, tentativa_erro, nao_resolveu, sem_tempo, outro"),
    "strategy_outro_texto": ("Texto escrito em 'Outro' na estratégia", "texto"),
    "resposta_do_pre": ("No pós: a resposta é a resposta certa do set do pré", "0/1; vazio no pré"),
    "igual_ao_proprio_pre": ("No pós: igual à própria resposta do aluno no pré", "0/1; vazio no pré"),
 
    # --- scores (escalas e mestra) ---
    "conhecimento": ("N.º de exercícios certos (cotação principal)", "0–3"),
    "conhecimento_estrito": ("N.º de exercícios certos (cotação estrita)", "0–3"),
    "conhecimento_amb_inv": ("N.º de exercícios certos (ambíguos invertidos)", "0–3"),
    "AP": ("PANAS: afeto positivo (soma de 5 itens)", "5–25"),
    "AN": ("PANAS: afeto negativo (soma de 5 itens)", "5–25"),
    "motivacao_intrinseca": ("SIMS: motivação intrínseca (média de 4 itens)", "1–7"),
    "regulacao_identificada": ("SIMS: regulação identificada (média de 4 itens)", "1–7"),
    "regulacao_externa": ("SIMS: regulação externa (média de 4 itens)", "1–7"),
    "desmotivacao": ("SIMS: desmotivação (média de 4 itens)", "1–7"),
 
    # --- SUS ---
    "SUS": ("System Usability Scale: [Σ(ímpares − 1) + Σ(5 − pares)] × 2,5", "0–100"),
 
    # --- jogo ---
    "game_results": ("Texto dos resultados do jogo colado pelo aluno (original)", "texto"),
    "game_finished": ("Terminou o jogo (laboratório salvo)", "1 = sim; 0 = explosão ou não terminou; vazio = sem dados"),
    "final_level": ("Nível em que o jogo terminou", "1, 2, 3"),
    "arrived_level3": ("Chegou ao nível 3 (diluições seriadas)", "0/1"),
    "xp": ("XP final (deduzido por erros na versão usada; não permite contar exercícios)", "número"),
    "estabilidade": ("Estabilidade final do laboratório", "%"),
    "game_time_min": ("Tempo de jogo da última tentativa (reinicia com recarga)", "minutos"),
    "lp_level1": ("N.º de exercícios do nível 1 em que o botão do LabPartner foi clicado", "0–3"),
    "lp_level2": ("N.º de exercícios do nível 2 em que o botão do LabPartner foi clicado", "0–3"),
    "lp_level3": ("N.º de exercícios do nível 3 em que o botão do LabPartner foi clicado", "0–3"),
    "lp_total": ("Soma de lp_level1–3", "0–9"),
    "lp_pct": ("lp_total / (3 × final_level) × 100 (mínimo, para quem recarregou)", "0–100 %"),
    "provavel_recarga": ("game_time_min < 5 (indicador descritivo)", "0/1"),
    "nota_jogo": ("Motivo da classificação manual do resultado do jogo", "texto"),
 
    # --- LabPartner durante o jogo (pós) ---
    "labpartner_function": ("Pergunta 'Durante o jogo, o LabPartner:', texto original (escolha múltipla)", "texto"),
    "lp_deu_codigo": ("LabPartner deu o código antes de resolver os exercícios", "0/1"),
    "lp_fez_calculos": ("LabPartner fez cálculos sem o aluno explicar o raciocínio", "0/1"),
    "lp_confuso": ("LabPartner deu respostas confusas ou contraditórias", "0/1"),
    "lp_funcionou_bem": ("LabPartner funcionou bem (nada de estranho)", "0/1"),
    "lp_problema": ("Pelo menos um problema pedagógico (código, cálculos, confuso)", "0/1"),
    "lp_outro_texto": ("Texto escrito em 'Outro'", "texto"),
    "lp_outro_cat": ("Codificação indutiva do 'Outro'", "problema_tecnico, nao_usou, outro"),
    "lp_problema_tecnico": ("'Outro': o LabPartner não funcionou / foi lento", "0/1"),
    "lp_nao_usou": ("'Outro': não usou o LabPartner", "0/1"),
    "lp_outro": ("'Outro' não codificado", "0/1"),

    # --- follow-up -----
    "more_practice": ("Voltou a praticar/estudar desde a aula (texto original)", "texto"),
    "practice_source": ("Como praticou", "exercicios_aula, plataforma, outros_recursos, nao"),
    "practiced": ("Praticou depois da aula", "0/1"),
    "perceived_usefulness": ("Considerou a sessão DILU-IA útil (item único, retrospetivo)", "1–5"),
    "tech_used": ("Dispositivos usados na sessão (texto original, escolha múltipla)", "texto"),
    "tech_phone": ("Usou telemóvel na sessão", "0/1"),
    "tech_tablet": ("Usou tablet na sessão", "0/1"),
    "tech_computer": ("Usou computador (portátil pessoal) na sessão", "0/1"),
    "tech_dont_remember": ("Não se lembra do dispositivo", "0/1"),
    "tech_n": ("N.º de tipos de dispositivo usados", "0–3"),
    "ai_use_raw": ("Acesso ao LabPartner na sessão (texto original, autorrelato retrospetivo)", "texto"),
    "ai_access": ("Acedeu ao LabPartner", "1 = acedeu; 0 = tentou sem conseguir; vazio = não tentou ou não se lembra"),
    "ai_use": ("Grau de uso do LabPartner", "2 = usou; 1 = quase não usou; 0 = não conseguiu ou não tentou; vazio = não se lembra"),
    "reason_not_ai_use": ("Motivo para não usar o LabPartner (texto original, escolha múltipla)", "texto"),
    "motivo_sozinho": ("Motivo: achou que conseguia resolver sozinho/a", "0/1"),
    "motivo_sem_acesso": ("Motivo: não conseguiu aceder", "0/1"),
    "motivo_sem_tempo": ("Motivo: não teve tempo", "0/1"),
    "motivo_ajuda_humana": ("Motivo: preferiu pedir ajuda aos docentes ou a colegas", "0/1"),
    "motivo_nao_percebi": ("Motivo: não percebeu como funcionava", "0/1"),
    "motivo_outro": ("Motivo em texto livre (críticas ao estilo de resposta do LabPartner)", "0/1"),
    "motivo_outro_texto": ("Texto livre do motivo", "texto"),
 
    # --- comentários (pós + follow) ---
    "general_comment": ("Comentários gerais sobre a experiência", "texto"),
    "positive_comment": ("Principais aspetos positivos do jogo", "texto"),
    "negative_comment": ("Principais aspetos negativos ou limitações do jogo", "texto"),
    "follow_comment": ("Comentário final do follow-up", "texto"),
}
 
# Famílias de itens: o número do item aponta para o texto
PANAS_ITENS = ["Interessado/a", "Nervoso/a", "Entusiasmado/a", "Amedrontado/a", "Inspirado/a",
               "Ativo/a", "Assustado/a", "Culpado/a", "Determinado/a", "Atormentado/a"]
SIMS_SUB_POR_ITEM = {i: sub for sub, itens in SIMS_SUBESCALAS.items()
                     for i in [int(x.split("_")[1]) for x in itens]}
SUS_POSITIVO = {1, 3, 5, 7, 9}

In [172]:
#Gerar o dicionário a partir das tabelas reais
def descrever(coluna):
    """Devolve (descrição, valores) para uma coluna, ou None se não estiver documentada."""
    if coluna.startswith("delta_pre_follow_"):
        base = descrever(coluna[len("delta_pre_follow_"):])
        return None if base is None else (f"Diferença follow-up − pré: {base[0]}", "follow-up − pré")
    if coluna.startswith("delta_pos_follow_"):
        base = descrever(coluna[len("delta_pos_follow_"):])
        return None if base is None else (f"Diferença follow-up − pós: {base[0]}", "follow-up − pós")    
    # sufixos de momento e prefixo delta_ (tabela larga)
    if coluna.startswith("delta_"):
        base = descrever(coluna[len("delta_"):])
        return None if base is None else (f"Diferença pós − pré: {base[0]}", "pós − pré")
    for sufixo, nome in [("_pre", "pré"), ("_pos", "pós"), ("_follow", "follow-up")]:
        if coluna.endswith(sufixo) and coluna[:-len(sufixo)] in DESCRICOES:
            desc, valores = DESCRICOES[coluna[:-len(sufixo)]]
            return (f"{desc} ({nome})", valores)
    # famílias de itens
    if re.fullmatch(r"affect_\d+", coluna):
        k = int(coluna.split("_")[1])
        tipo = "afeto positivo" if coluna in AP_ITENS else "afeto negativo"
        return (f"PANAS item {k}: {PANAS_ITENS[k - 1]} ({tipo})", "1–5")
    if re.fullmatch(r"motiv_\d+", coluna):
        k = int(coluna.split("_")[1])
        return (f"SIMS item {k} ({SIMS_SUB_POR_ITEM[k]})", "1–7")
    if re.fullmatch(r"sus\d+", coluna):
        k = int(coluna[3:])
        tipo = "positivo" if k in SUS_POSITIVO else "negativo"
        return (f"SUS item {k} ({tipo})", "1–5")
    return DESCRICOES.get(coluna)
 
TABELAS = {"respostas": respostas, "escalas": escalas, "participantes": participantes,
           "mestra": mestra, "larga": larga}
 
linhas = []
for nome_tabela, tabela in TABELAS.items():
    for coluna in tabela.columns:
        info = descrever(coluna)
        linhas.append({
            "tabela": nome_tabela,
            "variavel": coluna,
            "descricao": info[0] if info else "POR DOCUMENTAR",
            "valores": info[1] if info else "",
            "tipo_pandas": str(tabela[coluna].dtype),
            "n_validos": int(tabela[coluna].notna().sum()),
            "n_em_falta": int(tabela[coluna].isna().sum()),
        })
 
dicionario = pd.DataFrame(linhas)
 
por_documentar = dicionario[dicionario["descricao"] == "POR DOCUMENTAR"]
assert por_documentar.empty, f"Variáveis sem descrição:\n{por_documentar[['tabela', 'variavel']]}"


In [173]:
print(dicionario.groupby("tabela")["variavel"].count())
 
with pd.option_context("display.max_colwidth", 80, "display.max_rows", None):
    display(dicionario[dicionario["tabela"] == "participantes"])
 
dicionario.to_csv(PASTA / "dados_limpos" / "dicionario_variaveis.csv", index=False, encoding="utf-8-sig")
dicionario.to_excel(PASTA / "dados_limpos" / "dicionario_variaveis.xlsx", index=False)

tabela
escalas           34
larga            128
mestra            14
participantes     83
respostas         21
Name: variavel, dtype: int64


,tabela,variavel,descricao,valores,tipo_pandas,n_validos,n_em_falta
55,participantes,participant_id,"Código de participante (591/592 = respostas do pré dos inscritos 58 e 59, nã...",inteiro,int64,118,0
56,participantes,tem_pre,Tem questionário pré,0/1,int64,118,0
57,participantes,tem_pos,Tem questionário pós,0/1,int64,118,0
58,participantes,tem_follow,Tem questionário de follow-up,0/1,int64,118,0
59,participantes,par_pre_pos,Tem pré e pós emparelháveis (amostra da análise primária),0/1,int64,118,0
60,participantes,pre_pos_nao_emparelhavel,Código partilhado em simultâneo no pré (58/59); pré e pós não atribuíveis,0/1,int64,118,0
61,participantes,follow_nao_atribuivel,Submissões de follow-up com o código 113 de duas pessoas diferentes (provave...,0/1,int64,118,0
62,participantes,par_pre_follow,"Tem pré e follow-up atribuíveis (amostra principal da família 3, F1)",0/1,int64,118,0
63,participantes,tres_momentos,"Tem pré, pós e follow-up (amostra exploratória, F2)",0/1,int64,118,0
64,participantes,turno,"Turno da sessão de 21/09, atribuído pela hora de início do pré (ou do pós, s...","1, 2, 3",float64,115,3


In [174]:
fluxogram

{'pre_submissions': 119,
 'pos_submissions': 93,
 'follow_up_submissions': 49,
 'pos_fora_sessao': 2,
 'pre_ressubmissoes_removidas': 1,
 'codigos_reconciliados': 1,
 'nao_emparelhaveis': 2,
 'follow_ressubmissoes_removidas': 3,
 'follow_submissoes_repetidas_removidas': 2,
 'follow_nao_atribuiveis': 2,
 'inscritos': 118,
 'inscritos_sem_submissoes': 5,
 'pre_sem_consentimento': 5,
 'pos_sem_consentimento': 4,
 'pares_pre_pos': 87,
 'so_pre': 24,
 'so_pos': 0,
 'follow_com_pre': 41,
 'follow_com_pos': 32,
 'follow_3_momentos': 32,
 'follow_sem_pre_nem_pos': 1}

## 8. Logbook de alterações a este notebook

### >> 06/10/2026


##### v1: 06/10/2026 (`dados_limpos_2026-10-06`), antes do teste primário

**Identificação e consentimento.** A lista de inscritos (Forms cruzado com os consentimentos
em papel) tem 118 participantes. Foram excluídas as submissões sem consentimento (5 no pré
e 4 no pós). O código 113 do turno 3 passou a 133. A ressubmissão do 10 foi removida (fica a
primeira). As respostas simultâneas no pré com o código 59 passaram a 591 e 592 e, tal como os
pós do 58 e do 59, foram marcadas como não emparelháveis.

**Turnos e desvios.** O turno é atribuído pela hora de início do pré. O pós do 89 foi atribuído
ao turno 3. Os pós terminados mais de 15 minutos depois do fim do turno ficam marcados como
fora da sessão (89 e 5). Os 5 participantes que escreveram "sem tempo" no pós ficam marcados
em `pos_sem_tempo`.

**Jogo e LabPartner.** Os resultados do jogo foram extraídos do texto colado; 13 textos foram
classificados à mão (motivo em `nota_jogo`). As respostas "Outro" sobre o LabPartner foram
agrupadas em problema técnico, não usou e outro.

**Cotação.** Valor exato ± 0,5. As respostas sem vírgula decimal (perdida pelo Forms) foram
revistas valor a valor. As unidades são aceites no ex1 e no ex2, mas não no ex3, que pedia
unidade. Os casos ambíguos contam como certos e as respostas em branco como erradas. Há duas
versões para sensibilidade: estrita e com os ambíguos invertidos. Os métodos foram codificados
pelo texto da opção, e as respostas "Outro" por palavras-chave.

**Colaboração.** Ficam sinalizados os pares que partilham uma resposta invulgar rara (dada por
2 ou 3 alunos, excluindo os erros comuns), no mesmo turno, com 5 minutos ou menos entre
submissões. São 9 participantes.

**Escalas.** Os scores só são calculados com todos os itens respondidos (sem imputação).


### >> 11/10/2026

##### v2: 11/10/2026 (`dados_limpos_2026-10-11`), depois do teste primário

Acrescentadas variáveis descritivas de não resolução (respostas 0 ou "não sei", por instrução
da investigadora).

O código foi adaptado ao pandas 3.

A cotação não mudou: as colunas comuns à v1 foram verificadas como idênticas.

### >> 16/10/2026: Integração do follow-up e congelamento

In [175]:
ANTIGA = PASTA / "dados_limpos_2026-10-11"
CHAVE = ["participant_id", "momento", "exercicio"]

# Respostas: cotação do pré e do pós igual à congelada
a = pd.read_parquet(ANTIGA / "respostas.parquet")
n_ = respostas[respostas["momento"].astype(str).isin(["pre", "pos"])]
for t in (a, n_):
    t["momento"] = t["momento"].astype(str)
cols = ["correct", "correct_estrito", "correct_ambiguos_inv", "strategy", "volume_final", "nao_resolvido"]
comp = a[CHAVE + cols].merge(n_[CHAVE + cols], on=CHAVE, suffixes=("_antes", "_agora"), how="outer", indicator=True)
assert (comp["_merge"] == "both").all(), "Linhas do pré/pós diferentes"
for c in cols:
    assert comp[f"{c}_antes"].equals(comp[f"{c}_agora"]), f"Mudou: {c}"

# Larga: valores pré e pós e flags das sensibilidades iguais
a = pd.read_parquet(ANTIGA / "larga.parquet").set_index("participant_id")
n_ = larga.set_index("participant_id").loc[a.index]
for c in ["conhecimento_pre", "conhecimento_pos", "conhecimento_estrito_pre", "conhecimento_estrito_pos",
          "AP_pre", "AP_pos", "AN_pre", "AN_pos", "par_pre_pos", "suspeita_colaboracao",
          "pos_sem_tempo", "pos_fora_sessao", "game_finished", "SUS"]:
    assert a[c].equals(n_[c]), f"Mudou: {c}"
print("Pré e pós idênticos aos congelados a 11/10.")

Pré e pós idênticos aos congelados a 11/10.


**Dados**
- Follow-up real (`follow_up.xlsx`): 49 submissões, de 10/10 a 13/10 (fecho automático às 17h00, antes do reforço das 17h30). Todos responderam fora da aula, no seu próprio tempo (23 no último dia). Intervalo desde a sessão: 19–22 dias.
- Links enviados só a quem tinha consentimento assinado; todos os códigos estão na lista de inscritos.

**Limpeza**
- Ressubmissões da mesma pessoa em dias diferentes (16, 66, 147): fica a primeira (3 removidas).
- Código 67: três submissões idênticas com 5 s de intervalo, provavelmente submissão repetida por falha de ligação: fica a primeira (2 removidas).
- Código 113: duas submissões com respostas diferentes (provavelmente 113 e 133): códigos 1131 e 1132, não atribuíveis. Entram só nos descritivos.
- Resultado: 44 submissões válidas, 42 atribuíveis. 41 têm pré, 32 têm os três momentos e 1 não tem pré nem pós (esteve na sessão; entra só nos descritivos). O 58 e o 59 não responderam.

**Cotação e codificação**
- Mesmas regras do pré e do pós (F6). 559125 recuperado (2 casos). Ex3 com fator 1:10: erro de unidade e contagem de 2 diluições indistinguíveis (F7).
- Motivos para não usar o LabPartner codificados (opção real: "aos docentes ou a colegas"); 2 textos livres com críticas ao estilo de resposta → `motivo_outro`.
- Critério de colaboração aplicado ao follow-up sem a condição do turno (F13): 34 e 39 sinalizados. As flags do pré e do pós não mudaram.

**Estrutura**
- `nao_emparelhavel` mudou de nome para `pre_pos_nao_emparelhavel`.
- Novas flags: `tem_follow`, `follow_nao_atribuivel`, `par_pre_follow`, `tres_momentos`, `suspeita_colaboracao_follow`.
- Respostas, mestra e larga com o momento `follow`; diferenças `delta_pre_follow_*` e `delta_pos_follow_*`.

**Congelamento**
- Saída: `dados_limpos_2026-10-16`.
- Verificado: a cotação, os scores e as flags do pré e do pós são idênticos aos dados congelados a 11/10. Os resultados já obtidos não mudam.
- Daqui em diante só se mexe nos dados com uma nova entrada neste registo.